# LogSpace capacity analytics & forecast

**Input:** Excel master file `METRICS_XLSX`: rows of the daily `logspace_metrics.csv` produced by
`collect_logspace_metrics.sh`, appended by hand
(events per category, partition sizes from `backup_tool --cmd listpart`, `df` snapshots, rotation health).

**Export:** every chart is saved to `EXPORT_DIR` as an interactive `<chart>_<last complete day>.html`, every table
as `<table>_<last complete day>.xlsx` and as an interactive `<table>_table_<last complete day>.html`
(click a header to sort, type in the box to filter) — files of different days can be opened side by side and compared.
Each chart is followed by a short *How to read* note.

**Daily routine:** `Kernel → Restart & Run All`, then read **section 10 (Daily summary)** and **section 11 (compared with previous days)** first,
and go to the section a WARN/CRIT points to. Full methodology: `METHODOLOGY.md`.

| Section | Question it answers |
|---|---|
| 1 | Is the input data complete and fresh? |
| 2 | How many bytes does one event really cost (D)? |
| 3 | Do `listpart` bytes match the physical disk? |
| 3b | What left the disk since the previous snapshot? |
| 4 | Which categories grow and how fast? |
| 4.3 | How much of the flow is not normalized? |
| 4.4 | How few categories make most of the flow? |
| 4.5 | Which categories caused the last level change? |
| 4.6 | Which categories caused the spike days? |
| 4.7 | Which flow is driven by people, which by machines? |
| 4.8 | Are all categories arriving every day? (stability, silences) |
| 5 | Which categories are expensive on disk? |
| 6 | Model backtest: which forecast to trust |
| 7 | Daily volume forecast (bytes, EPS, events by category) |
| 8 | When rotation starts, retention depth, physical disk free |
| 9 | What-if: flow changes |
| 10 | Daily summary with statuses |
| 11 | Better or worse than yesterday and a week ago? |

## 0. Settings

In [ ]:
from pathlib import Path

METRICS_XLSX  = Path(r"I:\SHARE\AN\metrics.xlsx")  # master file: daily CSV rows appended by hand
METRICS_SHEET = 0           # sheet name or index (0 = first sheet)
LEGACY_XLSX   = None        # e.g. Path("eps.xlsx"): old Excel history (sheet RAW), merged under collector data
LEGACY_SHEET  = "RAW"

DBSPACE       = 0.85        # TailcutterDbspace as a fraction of total disk
TTL_DAYS      = 365         # TailcutterTtl
DISK_TOTAL_OVERRIDE = None  # bytes; None = take total from the latest df snapshot

ANOMALY_RATIO = 0.5         # day below 50% of rolling median = collection outage
TREND_WINDOW  = 90          # days of history for linear trends
BACKTEST_DAYS = 28          # holdout used to compare forecast models
HORIZON       = 180         # forecast horizon, days
VENDOR_D      = 200         # vendor formula coefficient for LogSpace, bytes/event
EXPORT_DIR    = Path("export")  # charts (.html) and tables (.xlsx) as <name>_<last complete day> (real data - never commit)

SARIMAX_WINDOW         = 180            # days of history SARIMAX is fitted on (None = all)
SARIMAX_ORDER          = (1, 1, 1)      # (p, d, q)
SARIMAX_SEASONAL_ORDER = (1, 0, 1, 7)   # (P, D, Q, period): weekly cycle

# Event analytics (sections 4.3-4.8)
NOT_NORMALIZED  = "Not_Normalized"  # collector label for events without a category
STEP_WINDOW     = 28        # days compared before / after a level change (4.5)
STEP_DATE       = None      # e.g. "2026-09-10" to analyse a known change; None = the largest change found
SPIKE_RATIO     = 0.25      # a day this much above its same-weekday baseline is a spike (4.6)
WEEKEND_MACHINE = 0.8       # weekend / weekday ratio at or above: background, machine-generated flow (4.7)
WEEKEND_HUMAN   = 0.5       # at or below: flow generated by people's working hours
SILENCE_RATIO   = 0.1       # a category below this share of its usual level is silent that day (4.8)

# Thresholds for the daily summary
WARN_DAYS_TO_LIMIT = 30     # rotation will start in fewer days -> WARN
WARN_VOLUME_DEV    = 0.30   # yesterday vs same-weekday baseline
WARN_CATEGORY_DEV  = 0.50   # per-category deviation vs same-weekday baseline
WARN_D_DRIFT       = 0.10   # D vs 28-day median
WARN_TC_LOG_AGE    = 90     # minutes since tailcutter log was written
WARN_NOT_NORMALIZED = 0.10  # share of events without a category

In [ ]:
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import minimize, nnls
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=RuntimeWarning)

GiB, TiB = 1024**3, 1024**4
DAY_MS = 86_400_000      # one day on a plotly date axis
REPORT_DAY = None        # last complete day, set in section 1; used in export file names
HOVER_DAY = "%d.%m.%Y, %a"


def gb(x):
    """Bytes -> GiB."""
    return x / GiB


def style(fig, title, ytitle, height=470, daily_ticks=False, slider=False):
    """Common look for charts over dates.
    The hover box shows the exact day (with weekday) and every line's value on it; a dotted vertical cursor
    follows the mouse. daily_ticks: a label under every day (short ranges). Otherwise labels get finer when
    you zoom in; slider adds a range slider and 1-month / 3-month / all buttons."""
    fig.update_layout(template="plotly_white", title=title, height=height, yaxis_title=ytitle,
                      hovermode="x unified", margin=dict(l=60, r=20, t=70, b=40),
                      legend=dict(orientation="h", yanchor="top", y=-0.3 if slider else -0.2))
    fig.update_xaxes(hoverformat=HOVER_DAY, showspikes=True, spikemode="across", spikesnap="cursor",
                     spikethickness=1, spikedash="dot", spikecolor="grey")
    if daily_ticks:
        xs = pd.DatetimeIndex([x for t in fig.data if t.x is not None and len(t.x) for x in (t.x[0], t.x[-1])])
        fig.update_xaxes(dtick=DAY_MS, tickformat="%d.%m", tickangle=-90,
                         range=[xs.min() - pd.Timedelta(hours=12), xs.max() + pd.Timedelta(hours=12)])
    else:
        fig.update_xaxes(tickformatstops=[dict(dtickrange=[None, 40 * DAY_MS], value="%d.%m"),
                                          dict(dtickrange=[40 * DAY_MS, None], value="%m.%Y")])
    if slider:
        fig.update_xaxes(rangeslider=dict(visible=True, thickness=0.06),
                         rangeselector=dict(buttons=[dict(count=1, label="1 mo", step="month", stepmode="backward"),
                                                     dict(count=3, label="3 mo", step="month", stepmode="backward"),
                                                     dict(step="all", label="all")]))
    return fig


def weekends(fig, index, row=None, col=None):
    """Shade Saturday-Sunday, so weekly dips can be matched to weekends at a glance.
    Skipped on long ranges (over 120 days), where the bands would turn into stripes."""
    index = pd.DatetimeIndex(index)
    if len(index) == 0 or (index.max() - index.min()).days > 120:
        return
    for d in index[index.dayofweek == 5]:
        fig.add_vrect(x0=d - pd.Timedelta(hours=12), x1=d + pd.Timedelta(hours=36), fillcolor="lightgrey",
                      opacity=0.3, line_width=0, layer="below", row=row, col=col)


def mark(fig, when, label="last complete day"):
    """Vertical line separating history from forecast (single-panel charts)."""
    fig.add_shape(type="line", x0=when, x1=when, y0=0, y1=1, yref="paper",
                  line=dict(color="black", dash="dot", width=1))
    fig.add_annotation(x=when, y=0.99, yref="paper", text=label, showarrow=False, xanchor="left", yanchor="top",
                       font=dict(size=11), bgcolor="rgba(255,255,255,0.7)")


def line(x, y, name, unit, **kw):
    """Line trace whose hover shows the value with its unit."""
    return go.Scatter(x=x, y=y, mode=kw.pop("mode", "lines"), name=name,
                      hovertemplate="%{y:,.1f} " + unit, **kw)


def weekday_baseline(df, weeks=4):
    """Per column: median of the same weekday over the previous `weeks` weeks (needs a continuous daily index)."""
    return pd.concat([df.shift(7 * k) for k in range(1, weeks + 1)]).groupby(level=0).median()


def palette(names):
    """Stable colour per name, the same on every chart."""
    from plotly.colors import qualitative
    pal = qualitative.Plotly + qualitative.D3 + qualitative.Set2
    return {n: pal[i % len(pal)] for i, n in enumerate(names)}


def rgba(hex_color, alpha):
    h = hex_color.lstrip("#")
    return f"rgba({int(h[0:2], 16)},{int(h[2:4], 16)},{int(h[4:6], 16)},{alpha})"


def out(fig, name):
    """Save the chart as EXPORT_DIR/<name>_<REPORT_DAY>.html and show it.
    plotly.js is written once next to the files, so they open offline in any browser."""
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    path = EXPORT_DIR / f"{name}_{REPORT_DAY:%Y-%m-%d}.html"
    fig.write_html(path, include_plotlyjs="directory")
    print(f"Saved: {path}")
    fig.show()


# ---- interactive tables: click a header to sort, type in the box to filter; plain JavaScript, no libraries ----
from html import escape as _esc
from uuid import uuid4
from IPython.display import HTML

# colours follow the editor theme (VS Code / JupyterLab variables), light fallback for a plain browser
_TABLE_CSS = """<style>
.itbl{font-family:Arial,sans-serif;font-size:13px;margin:6px 0;color:inherit}
.itbl-bar{margin-bottom:4px}.itbl-bar input{padding:2px 6px;width:240px}.itbl-bar span{opacity:.7;margin-left:8px}
.itbl-wrap{max-height:520px;overflow:auto;border:1px solid rgba(128,128,128,.4);display:inline-block;max-width:100%}
.itbl table{border-collapse:collapse;background:transparent;color:inherit}
.itbl tr{background:transparent !important}
.itbl th{position:sticky;top:0;padding:4px 8px;text-align:left;white-space:nowrap;cursor:pointer;color:inherit;
  background:var(--vscode-editor-background,var(--jp-layout-color2,#f3f5f7));border-bottom:1px solid rgba(128,128,128,.5)}
.itbl td{padding:3px 8px;border-top:1px solid rgba(128,128,128,.25);white-space:nowrap;text-align:left;color:inherit;background:transparent}
.itbl tbody tr:hover{background:rgba(128,128,128,.15) !important}
</style>"""

_TABLE_JS = r"""(function(){
var t=document.getElementById("T_@ID@"); if(!t) return;
var box=document.getElementById("F_@ID@"), cnt=document.getElementById("C_@ID@");
var body=t.tBodies[0], rows=Array.prototype.slice.call(body.rows), total=rows.length;
function key(s){ s=s.trim();
  var m=s.match(/^(\d{2})\.(\d{2})\.(\d{4})(?: (\d{2}):(\d{2}))?$/);
  if(m) return [0, +(m[3]+m[2]+m[1]+(m[4]||"00")+(m[5]||"00"))];
  var n=s.replace(/[\s%]/g,"");
  if(n!=="" && !isNaN(n)) return [0, parseFloat(n)];
  return [1, s.toLowerCase()]; }
var heads=t.tHead.rows[t.tHead.rows.length-1].cells;
Array.prototype.forEach.call(heads,function(th,i){
  th.title="click to sort";
  th.addEventListener("click",function(){
    var asc=th.getAttribute("data-dir")!=="asc";
    Array.prototype.forEach.call(heads,function(h){h.setAttribute("data-dir",""); h.textContent=h.textContent.replace(/ [▲▼]$/,"");});
    th.setAttribute("data-dir",asc?"asc":"desc"); th.textContent+=asc?" ▲":" ▼";
    rows.sort(function(a,b){ var x=key(a.cells[i].textContent), y=key(b.cells[i].textContent);
      if(x[0]!==y[0]) return x[0]-y[0]; var r=x[1]<y[1]?-1:(x[1]>y[1]?1:0); return asc?r:-r; });
    rows.forEach(function(r){ body.appendChild(r); }); }); });
box.addEventListener("input",function(){ var q=box.value.toLowerCase(), shown=0;
  rows.forEach(function(r){ var ok=r.textContent.toLowerCase().indexOf(q)>=0; r.style.display=ok?"":"none"; if(ok) shown++; });
  cnt.textContent=shown+" of "+total+" rows"; });
Array.prototype.forEach.call(heads,function(th,i){   // a column of only numbers / dates goes to the right
  var num=rows.every(function(r){ var v=r.cells[i].textContent; return v.trim()==="" || key(v)[0]===0; });
  if(num) rows.forEach(function(r){ r.cells[i].style.textAlign="right"; }); });
cnt.textContent=total+" rows";
})();"""


def _prep(df):
    """Index into columns, dates as dd.mm.yyyy[ hh:mm], numbers with a space as thousands separator."""
    d = df.copy()
    if not isinstance(d.index, pd.RangeIndex):
        d = d.reset_index()
    d.columns = [" / ".join(map(str, c)) if isinstance(c, tuple) else str(c) for c in d.columns]

    def cell(v):
        if v is None or (not isinstance(v, str) and pd.isna(v)):
            return ""
        if isinstance(v, pd.Timestamp):
            return v.strftime("%d.%m.%Y %H:%M" if (v.hour or v.minute) else "%d.%m.%Y")
        if isinstance(v, (float, np.floating)):
            return f"{v:,.2f}".replace(",", " ")
        if isinstance(v, (int, np.integer)) and not isinstance(v, bool):
            return f"{v:,}".replace(",", " ")
        return str(v)
    return d.apply(lambda col: col.map(cell))


def itable(df, colors=None, color_col=None, title=""):
    """HTML of an interactive table: sortable columns and a filter box.
    colors: {cell value: css} applied to column color_col (e.g. OK / WARN / CRIT)."""
    tid = uuid4().hex[:12]
    d = _prep(df)
    sty = d.style.hide(axis="index").set_uuid(tid)
    if colors and color_col in d.columns:
        smap = getattr(sty, "map", None) or sty.applymap      # pandas < 2.1 has only applymap
        sty = smap(lambda v: colors.get(v, ""), subset=[color_col])
    return (_TABLE_CSS +
            f'<div class="itbl"><div class="itbl-bar"><b>{_esc(title)}</b> '
            f'<input id="F_{tid}" placeholder="filter rows…"><span id="C_{tid}"></span></div>'
            f'<div class="itbl-wrap">{sty.to_html()}</div></div>'
            f'<script>{_TABLE_JS.replace("@ID@", tid)}</script>')


def show_table(df, colors=None, color_col=None, title=""):
    """Display a table as interactive HTML. Where the notebook front end does not run scripts, it shows as a plain table."""
    display(HTML(itable(df, colors, color_col, title)))


def out_table(df, name, show=True, colors=None, color_col=None):
    """Save the table as EXPORT_DIR/<name>_<REPORT_DAY>.xlsx and as an interactive
    <name>_table_<REPORT_DAY>.html (works in any browser), and display it."""
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    path = EXPORT_DIR / f"{name}_{REPORT_DAY:%Y-%m-%d}.xlsx"
    df.to_excel(path)
    title = f"{name} — {REPORT_DAY:%d.%m.%Y}"
    page = (f'<!doctype html><html><head><meta charset="utf-8"><title>{_esc(title)}</title></head><body>'
            f'{itable(df, colors, color_col, title)}</body></html>')
    hpath = EXPORT_DIR / f"{name}_table_{REPORT_DAY:%Y-%m-%d}.html"
    hpath.write_text(page, encoding="utf-8")
    print(f"Saved: {path}, {hpath.name}")
    if show:
        show_table(df, colors, color_col)

## 1. Load data and quality check

Duplicates (same `record_type`, `metric_date`, `key`) are resolved by the latest `collected_at`,
so re-running the collector is safe. Partitions deleted by rotation keep their last known size,
so the volume history survives rotation.

In [ ]:
raw_all = (pd.read_excel(METRICS_XLSX, sheet_name=METRICS_SHEET, dtype={"key": str})
             .dropna(subset=["record_type"]))
# Excel may turn date strings into real dates on paste; accept both
raw_all["collected_at"] = pd.to_datetime(raw_all["collected_at"], dayfirst=True, format="mixed")
raw_all["metric_date"] = pd.to_datetime(raw_all["metric_date"], dayfirst=True, format="mixed")

# Optional legacy Excel history: lowest priority, collector rows override it
if LEGACY_XLSX is not None:
    lx = pd.read_excel(LEGACY_XLSX, sheet_name=LEGACY_SHEET)
    lx = pd.DataFrame({
        "collected_at": pd.Timestamp("1970-01-01"),
        "metric_date": pd.to_datetime(lx["Date"]),
        "record_type": "events",
        "key": lx["category"].astype(str),
        "value": lx["total_events_per_cat_daily"],
    })
    raw_all = pd.concat([lx, raw_all], ignore_index=True)

raw = (raw_all.sort_values("collected_at")
              .drop_duplicates(["record_type", "metric_date", "key"], keep="last"))

# --- latest snapshot: what is stored right now ---
part_all = raw_all[raw_all["record_type"] == "partition"]
last_snap = part_all["collected_at"].max()
snap_day = last_snap.normalize()
stored = part_all[part_all["collected_at"] == last_snap]
stored_events = stored[stored["key"] == "events"].set_index("metric_date")["value"].sort_index()
stored_other = stored.loc[stored["key"] != "events", "value"].sum()
stored_total = stored["value"].sum()

# --- disk snapshots (time series by collection time) ---
disk = (raw_all[raw_all["record_type"] == "disk"]
        .pivot_table(index="collected_at", columns="key", values="value", aggfunc="last")
        .sort_index())
part_tot_by_snap = part_all.groupby("collected_at")["value"].sum()
disk["partitions"] = part_tot_by_snap.reindex(disk.index)
DISK_TOTAL = DISK_TOTAL_OVERRIDE or (disk["total"].iloc[-1] if len(disk) else None)
LIMIT = DISK_TOTAL * DBSPACE if DISK_TOTAL else None

health = (raw[raw["record_type"] == "health"].sort_values("collected_at")
          .groupby("key")["value"].last())

# --- events per category (date x category) ---
ev_cat = (raw[raw["record_type"] == "events"]
          .pivot_table(index="metric_date", columns="key", values="value", aggfunc="sum")
          .sort_index().fillna(0))

# --- daily frame: complete days only (the snapshot day itself is still being written) ---
part_hist = raw[(raw["record_type"] == "partition") & (raw["key"] == "events")] \
               .set_index("metric_date")["value"]
last_day = min(part_hist.index.max(), snap_day - pd.Timedelta(days=1))
first_day = part_hist.index.min()
daily = pd.DataFrame(index=pd.date_range(first_day, last_day, freq="D"))
daily.index.name = "date"
daily["bytes"] = part_hist.reindex(daily.index)
daily["events"] = ev_cat.sum(axis=1).reindex(daily.index)
daily["eps"] = daily["events"] / 86400
daily["D"] = daily["bytes"] / daily["events"]

roll_med = daily["bytes"].rolling(15, center=True, min_periods=5).median()
daily["anomaly"] = daily["bytes"].isna() | (daily["bytes"] < roll_med * ANOMALY_RATIO)

# Clean continuous series for models: outage days are interpolated, not dropped
y_bytes = daily["bytes"].mask(daily["anomaly"]).interpolate(limit_direction="both")
REPORT_DAY = last_day

# --- quality report ---
missing_bytes = daily.index[daily["bytes"].isna()]
missing_events = daily.index[daily["events"].isna()]
print(f"Latest snapshot:      {last_snap:%d.%m.%Y %H:%M}")
print(f"Daily history:        {first_day:%d.%m.%Y} … {last_day:%d.%m.%Y} ({len(daily)} days)")
print(f"Stored now:           {len(stored_events)} event partitions, {stored_total/TiB:.2f} TiB")
print(f"Events data:          {daily['events'].notna().sum()} days, {ev_cat.shape[1]} categories")
print(f"Disk snapshots:       {len(disk)}")
print(f"Days without bytes:   {len(missing_bytes)}")
print(f"Days without events:  {len(missing_events)}  (backfill with: collect_logspace_metrics.sh -f … -t …)")
print("Outage days (excluded):", ", ".join(f"{d:%d.%m.%Y}" for d in daily.index[daily["anomaly"] & daily["bytes"].notna()]) or "none")

# --- daily volume: one bar per day (size of that day's events partition, latest known value) ---
real = ~daily["anomaly"]
fig = go.Figure()
fig.add_trace(go.Bar(x=daily.index[real], y=gb(daily.loc[real, "bytes"]), name="GiB per day",
                     marker_color="steelblue", hovertemplate="%{y:,.1f} GiB"))
fig.add_trace(go.Bar(x=daily.index[~real], y=gb(daily.loc[~real, "bytes"].fillna(0)), name="outage day (excluded)",
                     marker_color="crimson", hovertemplate="%{y:,.1f} GiB"))
fig.add_trace(line(daily.index, gb(y_bytes.rolling(7).mean()), "7-day mean", "GiB", line=dict(color="black")))
weekends(fig, daily.index)
style(fig, "Daily volume: size of each day's events partition", "GiB/day", slider=True)
out(fig, "daily_volume")

daily_tbl = pd.DataFrame({"GiB": gb(daily["bytes"]).round(2), "events": daily["events"],
                          "EPS": daily["eps"].round(0), "D, B/event": daily["D"].round(1),
                          "outage": daily["anomaly"]})
out_table(daily_tbl, "daily", show=False)

### How to read section 1

**The printed report**
- `Latest snapshot` — time of the newest collector run. Older than a day → the collector did not run (section 10).
- `Daily history` — days with partition sizes. `Stored now` — partitions on disk at the latest snapshot.
- `Events data` — days with per-category event counts. Sections 2, 4, 5 and the EPS charts need these; if they are
  missing, backfill them with `collect_logspace_metrics.sh -f … -t …`.
- `Outage days` — days below `ANOMALY_RATIO` of the rolling median. They are excluded from models and drawn red.

**The chart** — one bar per day = size of that day's events partition, GiB.
- Blue bars are real days, red bars are outage days (excluded), the black line is the 7-day mean.
- Grey bands are weekends: the regular dips should fall inside them.
- Hover shows the exact date with the weekday; use *1 mo / 3 mo / all* or the slider under the chart to zoom.
- What to look for: the level of the black line (today's typical day), steps in it (a source was added or lost),
  single tall bars (one-off events) and red bars (collection problems).

**The table** `daily_<day>.xlsx` (saved, not shown): per day — GiB, events, EPS, D and the outage flag.

*Example (illustrative):* the black line runs flat at ~40 GiB/day, then within a week climbs to ~60 and stays there
→ a step, not a trend: find which categories grew in section 4. A single 90 GiB bar on a Tuesday with normal days
around it is a one-off event; a red bar at 5 GiB on a Saturday is a collection outage, not a quiet weekend.

## 2. Real cost of one event (D)

**Left:** daily bytes vs daily events with a linear regression.
The slope is the marginal cost of one extra event; a tight cloud along the line means
"disk usage is driven by event count" and EPS-based forecasts are reliable.
**Right:** D over time. A step in D with flat EPS means the event mix changed
(heavier categories or longer raw messages), not the flow.

**When D is computed.** At least **7** usable days are needed. A day is usable when it is complete
(before the latest snapshot day), has `events` partition bytes from `listpart`, has events data and is
not an outage day. `D_FACT` is the median D over the last 28 usable days; with fewer than 28 it is
less stable (weekdays and weekends differ). With fewer than 7 days `D_FACT` falls back to `VENDOR_D`.

**Vendor value.** `VENDOR_D = 200` bytes/event: coefficient of the LogSpace sizing formula in the
PT MaxPatrol SIEM documentation (exact document version and section not recorded here).

In [ ]:
dd = daily.loc[~daily["anomaly"] & daily["events"].notna()]
HAS_D = len(dd) >= 7

if HAS_D:
    slope, icpt = np.polyfit(dd["events"], dd["bytes"], 1)
    r2 = np.corrcoef(dd["events"], dd["bytes"])[0, 1] ** 2
    D_FACT = dd["D"].tail(28).median()

    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08,
                        subplot_titles=("Bytes vs events (one dot = one day)", "D over time"))
    fig.add_trace(go.Scatter(x=dd["events"] / 1e6, y=gb(dd["bytes"]), mode="markers", name="days",
                             customdata=dd.index.strftime(HOVER_DAY),
                             hovertemplate="%{customdata}<br>%{x:,.1f} M events<br>%{y:,.1f} GiB<extra></extra>"),
                  row=1, col=1)
    xs = np.linspace(dd["events"].min(), dd["events"].max(), 50)
    fig.add_trace(go.Scatter(x=xs / 1e6, y=gb(icpt + slope * xs), mode="lines", line=dict(color="red"),
                             name=f"fit: {slope:.0f} B/event, R²={r2:.3f}", hoverinfo="skip"), row=1, col=1)
    fig.add_trace(go.Scatter(x=xs / 1e6, y=gb(VENDOR_D * xs), mode="lines", line=dict(color="black", dash="dash"),
                             name=f"vendor: {VENDOR_D} B/event", hoverinfo="skip"), row=1, col=1)
    fig.add_trace(go.Scatter(x=dd.index, y=dd["D"], mode="markers", name="daily D",
                             hovertemplate="%{x|" + HOVER_DAY + "}<br>%{y:,.0f} B/event<extra></extra>"), row=1, col=2)
    fig.add_trace(go.Scatter(x=dd.index, y=dd["D"].rolling(14, min_periods=5).median(), mode="lines",
                             line=dict(color="red"), name="D, 14-day median",
                             hovertemplate="%{x|" + HOVER_DAY + "}<br>%{y:,.0f} B/event<extra>14-day median</extra>"),
                  row=1, col=2)
    fig.add_hline(y=VENDOR_D, line_dash="dash", line_color="black", row=1, col=2)
    weekends(fig, dd.index, row=1, col=2)
    fig.update_xaxes(title_text="events per day, millions", row=1, col=1)
    fig.update_yaxes(title_text="GiB per day", row=1, col=1)
    fig.update_yaxes(title_text="bytes per event", row=1, col=2)
    fig.update_xaxes(row=1, col=2, hoverformat=HOVER_DAY,
                     tickformatstops=[dict(dtickrange=[None, 40 * DAY_MS], value="%d.%m"),
                                      dict(dtickrange=[40 * DAY_MS, None], value="%m.%Y")])
    fig.update_layout(template="plotly_white", height=470, hovermode="closest",
                      legend=dict(orientation="h", yanchor="top", y=-0.2))
    out(fig, "cost_of_event")

    print(f"D (last 28 days median): {D_FACT:.1f} B/event; vendor formula overestimates by x{VENDOR_D / D_FACT:.2f}")
else:
    D_FACT = VENDOR_D
    print("Not enough days with events data; D falls back to the vendor value")

### How to read section 2

**Left — bytes vs events.** One dot = one day: X — events that day, Y — GiB on disk. Hover a dot to see its date.
- Dots lying close along the red line (R² close to 1) mean disk usage is driven by the number of events:
  forecasts through EPS are reliable, and the slope of the red line is the cost of one extra event.
- The gap between the red line and the black dashed one is how far the vendor's sizing formula is from reality.
- Dots far above the line are days with unusually heavy events; far below — incomplete days.

**Right — D over time** (bytes per event per day, red = 14-day median).
- Flat D with changing EPS — the flow changes, the event mix does not.
- A step in D with flat EPS — the event mix changed (heavier categories, longer raw messages), not the flow.
- Weekend dots may sit apart from weekdays: a different mix of sources at the weekend.

*Example (illustrative):* R² = 0.97, fit 95 B/event, vendor 200 → disk usage follows the event count and the vendor
formula overestimates disk about twice. D steps from ~95 to ~120 in one week while EPS stays flat → a heavy source was
added; check section 5 for the category whose cost grew.

## 3. Listpart bytes vs physical disk

`df used / sum(listpart bytes)` per snapshot. A stable ratio close to 1 means `listpart`
reflects real disk usage; the notebook uses this ratio to convert partition forecasts into physical free space.
A growing ratio means something other than partitions is eating the disk (staging, metadata, other files).

In [ ]:
if len(disk) and disk["partitions"].notna().any():
    disk["ratio"] = disk["used"] / disk["partitions"]
    RATIO = disk["ratio"].dropna().iloc[-1]

    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08,
                        subplot_titles=("Physical disk", "df used / sum of partitions"))
    hov = "%{x|%d.%m.%Y %H:%M}<br>%{y:,.2f} TiB<extra>%{fullData.name}</extra>"
    fig.add_trace(go.Scatter(x=disk.index, y=disk["used"] / TiB, mode="lines+markers", name="df used",
                             hovertemplate=hov), row=1, col=1)
    fig.add_trace(go.Scatter(x=disk.index, y=disk["partitions"] / TiB, mode="lines+markers",
                             name="sum of partitions", hovertemplate=hov), row=1, col=1)
    fig.add_hline(y=DISK_TOTAL / TiB, line_color="black", line_width=1, row=1, col=1,
                  annotation_text="disk total", annotation_position="top left")
    if LIMIT:
        fig.add_hline(y=LIMIT / TiB, line_color="red", line_dash="dash", row=1, col=1,
                      annotation_text=f"rotation limit {DBSPACE:.0%}", annotation_position="bottom left")
    fig.add_trace(go.Scatter(x=disk.index, y=disk["ratio"], mode="lines+markers", name="ratio",
                             hovertemplate="%{x|%d.%m.%Y %H:%M}<br>%{y:.3f}<extra>ratio</extra>"), row=1, col=2)
    fig.update_yaxes(title_text="TiB", row=1, col=1)
    fig.update_yaxes(range=[0.9, max(1.2, disk["ratio"].max() * 1.05)], row=1, col=2)
    fig.update_xaxes(tickformat="%d.%m", dtick=DAY_MS if len(disk) <= 60 else None)
    fig.update_layout(template="plotly_white", height=450, hovermode="closest",
                      legend=dict(orientation="h", yanchor="top", y=-0.15))
    out(fig, "disk")
    print(f"Latest: df used {disk['used'].iloc[-1]/TiB:.2f} TiB, partitions {disk['partitions'].iloc[-1]/TiB:.2f} TiB, ratio {RATIO:.3f}")
else:
    RATIO = 1.0
    print("No disk snapshots yet: ratio assumed 1.0")

### How to read section 3

One point = one collector run (snapshot), so the axis is snapshot time, not days of events.

**Left — physical disk.**
- `df used` (what the file system reports) and `sum of partitions` (what `listpart` reports) should run close to
  each other; both under the black *disk total* line.
- The red dashed line is the rotation limit from `DBSPACE`. **If the real partitions stay steadily above it, either
  rotation is not working or `DBSPACE` here differs from the tailcutter setting on the server** — check both.
- A flat `df used` while new days keep arriving means something deletes as much as is written every day: rotation works,
  just at the level where the line sits.

**Right — the ratio df used / partitions.**
- Stable and close to 1.00–1.05: `listpart` reflects real disk usage, and forecasts of partitions are forecasts of the disk.
- Growing: something other than partitions is filling the disk (staging, metadata, foreign files) — tailcutter will not
  remove it.

*Example (illustrative):* df used 9.4 TiB, partitions 9.3 TiB, ratio 1.01 and flat for a month → `listpart` can be
trusted. Ratio drifting 1.02 → 1.10 in two weeks → look for large files outside the partitions on the data volume.

### 3b. Rotation log: what left the disk between snapshots

Every snapshot holds the full `listpart` list, so two neighbouring snapshots show exactly which partitions
disappeared and how big they were. **Needs a snapshot every day**: after a gap, the deletions of the whole gap are
summed into one row.

**The chart** — one group per snapshot: blue up = GiB added since the previous snapshot (yesterday's partition
plus what was still being written), red down = GiB deleted; black line = days of events stored (right axis).
Hover a red bar to see which days were deleted.
- Added ≈ deleted every day — the disk is in balance: rotation deletes as much as arrives.
- Red bars stop while blue continue — rotation stopped; the disk is filling (see section 8).
- The black line falling — retention depth is shrinking: more is deleted than a day's worth.

**The table** `rotation_log` (last 10 rows shown): deleted days, GiB freed and added, oldest day, days stored, disk use
at that snapshot and the reason:
- `TTL` — the deleted days were older than `TTL_DAYS`;
- `size limit` — younger than TTL: deleted because the disk reached the rotation level;
- **`NOT THE OLDEST — manual delete?`** — a day was removed while older days stayed. Tailcutter deletes from the oldest
  end only, so this is a manual `deletepart` or a fault — find out who and why.

Under the table: the **observed rotation level** — the median disk use at snapshots where something was deleted.
This is the real threshold tailcutter works at; if it differs from `DBSPACE` in section 0, set `DBSPACE` to it,
otherwise every capacity forecast uses the wrong limit.

*Example (illustrative):* every day 1 day deleted (~45 GiB), ~55 GiB added, days stored 241 → 240 → 239 — retention
shrinks by a day per day; deletions happen at 92 % disk use while `DBSPACE` says 85 % → correct `DBSPACE` to 0.92.

In [ ]:
# --- 3b. rotation log: what left the disk between snapshots ---
ROT, ROT_LEVEL = None, None
snaps = sorted(part_all["collected_at"].unique())
if len(snaps) >= 2:
    by_snap = {s: part_all[part_all["collected_at"] == s].groupby(["metric_date", "key"])["value"].sum() for s in snaps}
    rows = []
    for prev, cur in zip(snaps[:-1], snaps[1:]):
        p_, c_ = by_snap[prev], by_snap[cur]
        gone = p_.index.difference(c_.index)
        gone_days = sorted({d for d, _ in gone})
        ev_c = c_[c_.index.get_level_values("key") == "events"].droplevel("key")
        snap_day = pd.Timestamp(cur).normalize()
        not_oldest = any((ev_c.index < d).any() for d in gone_days)
        reason = ("" if not gone_days else "NOT THE OLDEST — manual delete?" if not_oldest
                  else "TTL" if min((snap_day - d).days for d in gone_days) > TTL_DAYS else "size limit")
        freed = p_.loc[gone].sum() if len(gone) else 0.0
        rows.append({"snapshot": pd.Timestamp(cur),
                     "hours since previous": round((pd.Timestamp(cur) - pd.Timestamp(prev)).total_seconds() / 3600, 1),
                     "days deleted": len(gone_days), "deleted": ", ".join(f"{d:%d.%m.%Y}" for d in gone_days),
                     "freed, GiB": round(gb(freed), 1), "added, GiB": round(gb(c_.sum() - p_.sum() + freed), 1),
                     "net, GiB": round(gb(c_.sum() - p_.sum()), 1),
                     "oldest day": ev_c.index.min() if len(ev_c) else pd.NaT, "days stored": len(ev_c),
                     "reason": reason})
    ROT = pd.DataFrame(rows).set_index("snapshot")
    if len(disk):
        ROT["disk used, %"] = (disk["used"] / disk["total"] * 100).reindex(ROT.index).round(1)
        lv = ROT.loc[ROT["days deleted"] > 0, "disk used, %"].dropna()
        ROT_LEVEL = lv.median() if len(lv) else None

    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Bar(x=ROT.index, y=ROT["added, GiB"], name="added, GiB", marker_color="steelblue",
                         hovertemplate="+%{y:,.1f} GiB"), secondary_y=False)
    fig.add_trace(go.Bar(x=ROT.index, y=-ROT["freed, GiB"], name="deleted, GiB", marker_color="crimson",
                         customdata=ROT["deleted"], hovertemplate="%{y:,.1f} GiB: %{customdata}"), secondary_y=False)
    fig.add_trace(go.Scatter(x=ROT.index, y=ROT["days stored"], name="days stored", mode="lines+markers",
                             line=dict(color="black"), hovertemplate="%{y} days stored"), secondary_y=True)
    style(fig, "Rotation log: added vs deleted between snapshots", "GiB", daily_ticks=len(ROT) <= 60,
          slider=len(ROT) > 60)
    fig.update_layout(barmode="relative")
    fig.update_xaxes(hoverformat="%d.%m.%Y %H:%M")
    fig.update_yaxes(title_text="days stored", secondary_y=True)
    out(fig, "rotation_log")
    out_table(ROT, "rotation_log", show=False)
    show_table(ROT.tail(10), title="last 10 snapshots")
    if ROT_LEVEL is not None:
        print(f"Observed rotation level: deletions happen at ~{ROT_LEVEL:.1f} % disk use "
              f"(median of {len(lv)} snapshots with deletions); DBSPACE in section 0 = {DBSPACE:.0%}")
    else:
        print("No deletions between snapshots yet — the observed rotation level is unknown")
else:
    print("The rotation log needs at least two snapshots")

## 4. Category dynamics

**Top:** share of each category in the daily flow. **Bottom:** linear trend per category
over the last `TREND_WINDOW` days, as % per month. Fast growers are the main drivers of future disk usage;
sudden drops usually mean a source stopped sending.

In [ ]:
ec = ev_cat.reindex(daily.index).loc[~daily["anomaly"]].dropna(how="all")
HAS_CAT = len(ec) >= 14

if HAS_CAT:
    share = ec.sum() / ec.sum().sum()
    main = share[share >= 0.01].sort_values(ascending=False).index
    grp = ec[main].copy()
    grp["other <1%"] = ec.drop(columns=main).sum(axis=1)

    fig = go.Figure()
    for col in grp.columns:
        fig.add_trace(go.Scatter(x=grp.index, y=grp[col], name=col, mode="lines", stackgroup="one",
                                 groupnorm="percent", line=dict(width=0.5),
                                 hovertemplate="%{y:.1f} %"))
    style(fig, "Category share of daily events", "% of daily events", height=520, slider=True)
    out(fig, "category_share")

    w = grp.tail(TREND_WINDOW)
    t = (w.index - w.index[0]).days.values
    trend = {}
    for c in tqdm(w.columns, desc="category trends"):
        k, _ = np.polyfit(t, w[c].values, 1)
        trend[c] = {"share_%": share.get(c, w[c].sum() / w.sum().sum()) * 100,
                    "growth_%/month": k * 30 / max(w[c].mean(), 1) * 100,
                    "Δevents/day per month": k * 30}
    trend = pd.DataFrame(trend).T.sort_values("growth_%/month")

    fig = go.Figure(go.Bar(x=trend["growth_%/month"], y=trend.index, orientation="h",
                           marker_color=np.where(trend["growth_%/month"] > 0, "crimson", "seagreen"),
                           customdata=np.c_[trend["share_%"], trend["Δevents/day per month"]],
                           hovertemplate="%{y}<br>growth %{x:+.1f} % per month<br>share %{customdata[0]:.1f} %"
                                         "<br>%{customdata[1]:+,.0f} events/day per month<extra></extra>"))
    fig.update_layout(template="plotly_white", height=max(380, 26 * len(trend)), xaxis_title="% per month",
                      title=f"Linear trend per category, last {len(w)} days ({w.index[0]:%d.%m.%Y} – {w.index[-1]:%d.%m.%Y})")
    out(fig, "category_growth")
    out_table(trend.round(1).sort_values("Δevents/day per month", ascending=False), "category_trends")
else:
    print("Not enough events data (need ≥14 days)")

### How to read section 4

**Category share** — each band is one category's share of the day's events, all bands add up to 100 %.
Hover a day to see every category's share on it; click a category in the legend to hide it, double-click to show it alone.
- A band that slowly widens — a category growing faster than the rest.
- A band that suddenly appears, jumps or disappears — a source was connected, started flooding or stopped sending.
  A disappearing band is the earliest sign of a collection problem.
- Categories under 1 % are summed into `other <1%`.

**Linear trend per category** — growth of each category over the last `TREND_WINDOW` days, % per month
(red = growing, green = shrinking). Hover a bar for its share and the absolute change in events per day.
Read it together with the share: +30 % per month of a 0.5 % category matters less than +5 % of a 40 % one.

**The table** `category_trends` — share, growth % per month and Δ events/day per month, sorted by the absolute change:
the top rows are the categories that drive the future flow.

*Example (illustrative):* Operating system keeps 40 % share but grows +6 % per month (+2 M events/day each month) — the
main driver. Proxy grows +25 % per month but has a 2 % share — watch, but it will not decide capacity.
A Firewall band that drops from 20 % to 3 % in one day → a firewall stopped sending; check its collection task.

### 4.3 Events without a category

Events the SIEM stored without `event_src_category` — the collector writes them as `Not_Normalized`. They take
disk space but give correlation rules almost nothing to work with. **Bars** — such events per day,
**grey line** — their share of all events per day, **black line** — the same as a 7-day mean (right axis);
the share rises at weekends when people-driven categories go quiet, so read the black line.
- The level of the line is a quality metric of normalization: the lower, the better.
- A step up — a source was connected without a working normalization rule (or a rule broke after an update).
- A step down — a normalization fix worked, e.g. after an expertise update or a product upgrade.

*Example (illustrative):* the share sits at 12 % for months, then jumps to 20 % on one day and stays there → a new
source arrives unparsed; find it in the SIEM by the date and fix normalization before it costs months of disk.

In [ ]:
# --- 4.3 events without a category ---
# Daily events per category with outage days blanked; reused by 4.3-4.8
evd = ev_cat.reindex(daily.index)
evd.loc[daily["anomaly"].values] = np.nan
NN_SHARE = None
if HAS_CAT and NOT_NORMALIZED in evd.columns:
    total_ev = evd.sum(axis=1, min_count=1)
    nn = evd[NOT_NORMALIZED]
    nn_share = nn / total_ev * 100
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Bar(x=nn.index, y=nn / 1e6, name="events without a category, M/day", marker_color="indianred",
                         hovertemplate="%{y:,.1f} M"), secondary_y=False)
    fig.add_trace(go.Scatter(x=nn_share.index, y=nn_share, mode="lines", name="share of all events, %",
                             line=dict(color="grey", width=1), hovertemplate="%{y:.1f} %"), secondary_y=True)
    fig.add_trace(go.Scatter(x=nn_share.index, y=nn_share.rolling(7, min_periods=4).mean(), mode="lines",
                             name="share, 7-day mean", line=dict(color="black", width=2.5),
                             hovertemplate="%{y:.1f} %"), secondary_y=True)
    weekends(fig, nn.dropna().index)
    style(fig, "Events without a category (not normalized)", "M events/day", slider=True)
    fig.update_yaxes(title_text="% of all events", rangemode="tozero", secondary_y=True)
    out(fig, "not_normalized")
    last28 = nn_share.dropna().tail(28)
    NN_SHARE = last28.mean() / 100
    print(f"Without a category over the last {len(last28)} days: {last28.mean():.1f} % of events "
          f"({nn.dropna().tail(28).mean() / 1e6:,.1f} M events/day); min {last28.min():.1f} %, max {last28.max():.1f} %")
elif HAS_CAT:
    print(f"No '{NOT_NORMALIZED}' category in the data")
else:
    print("Not enough events data (need ≥14 days)")

### 4.4 Concentration

Categories sorted by their share of events over the last 28 days (bars) and the running total (black line, right
axis); the red dotted line is 80 %. The title says how many categories make 80 % of the flow.
- If a handful of categories make 80 %, every optimisation discussion is about them only; the long tail does not
  matter for capacity, however many categories it has.
- The table `category_concentration` gives events per day, share and the running total.

*Example (illustrative):* 3 of 18 categories make 80 % — Operating system 45 %, Firewall 22 %, events without a
category 14 %. Cutting a small category by half changes nothing; cutting 10 % of Operating system saves more than
removing the eight smallest categories entirely.

In [ ]:
# --- 4.4 concentration: how few categories make most of the flow ---
if HAS_CAT:
    recent = evd.dropna(how="all").tail(28).mean().sort_values(ascending=False)
    share_r = recent / recent.sum() * 100
    cum = share_r.cumsum()
    N80 = int((cum < 80).sum()) + 1
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Bar(x=share_r.index, y=share_r, name="share of events, %", marker_color="steelblue",
                         customdata=recent.values / 1e6,
                         hovertemplate="%{x}<br>%{y:.1f} % (%{customdata:,.1f} M events/day)<extra></extra>"),
                  secondary_y=False)
    fig.add_trace(go.Scatter(x=cum.index, y=cum, mode="lines+markers", name="cumulative, %", line=dict(color="black"),
                             hovertemplate="%{x}<br>cumulative %{y:.1f} %<extra></extra>"), secondary_y=True)
    fig.add_hline(y=80, line_dash="dot", line_color="red", secondary_y=True)
    fig.update_layout(template="plotly_white", height=470, hovermode="closest",
                      title=f"Concentration: {N80} of {len(share_r)} categories make 80 % of events (last 28 days)",
                      legend=dict(orientation="h", yanchor="top", y=-0.35))
    fig.update_yaxes(title_text="% of events", secondary_y=False)
    fig.update_yaxes(title_text="cumulative %", range=[0, 105], secondary_y=True)
    out(fig, "category_concentration")
    out_table(pd.DataFrame({"M events/day": (recent / 1e6).round(2), "share, %": share_r.round(2),
                            "cumulative, %": cum.round(2)}), "category_concentration")
else:
    print("Not enough events data (need ≥14 days)")

### 4.5 Level change: which categories caused it

The notebook looks for the day where the average flow of the next `STEP_WINDOW` days differs most from the previous
`STEP_WINDOW` days (or uses `STEP_DATE` if you set it), and splits that difference by category.
- **First chart** — total events per day with the two compared windows shaded and the change date.
- **Second chart** (waterfall) — how much each category added (red) or removed (green); the last bar is the total.
- **Table** `level_change_by_category` — before / after per day, the change and each category's share of it.

How to use it: a change made by one or two categories is a source connected or lost — a new permanent level, so the
forecast should start from it. A change spread evenly over all categories is general growth of activity.

*Example (illustrative):* change ~12.09: +180 M events/day (+28 %); Proxy +120 M (67 % of the change), Operating
system +40 M, the rest small → a proxy started sending much more on that day; check what was changed there and whether
all of it is needed.

In [ ]:
# --- 4.5 which categories caused the level change ---
LEVEL_CHANGE = None
tot = evd.sum(axis=1, min_count=1)
W = STEP_WINDOW
if HAS_CAT and tot.notna().sum() >= 2 * W:
    v = tot.values.astype(float)
    if STEP_DATE:
        i_best = int(daily.index.get_indexer([pd.Timestamp(STEP_DATE)])[0])
    else:
        gains = {}
        for i in range(W, len(v) - W + 1):
            b, a = v[i - W:i], v[i:i + W]
            if np.isfinite(b).sum() >= 0.7 * W and np.isfinite(a).sum() >= 0.7 * W:
                gains[i] = np.nanmean(a) - np.nanmean(b)
        i_best = max(gains, key=lambda k: abs(gains[k])) if gains else None
    if i_best is not None and i_best >= W:
        d0 = daily.index[i_best]
        before, after = evd.iloc[i_best - W:i_best].mean(), evd.iloc[i_best:i_best + W].mean()
        delta = (after - before).fillna(0).sort_values(key=abs, ascending=False)
        d_tot, b_tot = delta.sum(), before.sum()
        LEVEL_CHANGE = (d0, d_tot, d_tot / b_tot if b_tot else np.nan)

        # chart 1: the total with the two compared windows
        fig = go.Figure()
        fig.add_vrect(x0=daily.index[i_best - W], x1=d0, fillcolor="lightgrey", opacity=0.35, line_width=0,
                      annotation_text=f"before: {W} days", annotation_position="bottom left")
        fig.add_vrect(x0=d0, x1=daily.index[min(i_best + W, len(daily) - 1)], fillcolor="lightskyblue", opacity=0.3,
                      line_width=0, annotation_text=f"after: {W} days", annotation_position="bottom right")
        fig.add_trace(line(tot.index, tot / 1e6, "events per day", "M", line=dict(color="lightgrey")))
        fig.add_trace(line(tot.index, tot.rolling(7, min_periods=4).mean() / 1e6, "7-day mean", "M",
                           line=dict(color="black")))
        mark(fig, d0, f"change ~{d0:%d.%m.%Y}")
        style(fig, f"Level change ~{d0:%d.%m.%Y}: {d_tot / 1e6:+,.1f} M events/day ({d_tot / b_tot:+.0%})",
              "M events/day", slider=True)
        out(fig, "level_change")

        # chart 2: contribution of each category to the change
        top = delta.head(10)
        rest = delta.iloc[10:].sum()
        xs = list(top.index) + (["all other"] if len(delta) > 10 else []) + ["total"]
        ys = list(top.values / 1e6) + ([rest / 1e6] if len(delta) > 10 else []) + [d_tot / 1e6]
        fig = go.Figure(go.Waterfall(x=xs, y=ys, measure=["relative"] * (len(xs) - 1) + ["total"],
                                     increasing=dict(marker_color="crimson"), decreasing=dict(marker_color="seagreen"),
                                     totals=dict(marker_color="steelblue"),
                                     hovertemplate="%{x}<br>%{y:+,.2f} M events/day<extra></extra>"))
        fig.update_layout(template="plotly_white", height=460, yaxis_title="Δ M events/day",
                          title=f"Who made the change: {W} days after vs {W} days before {d0:%d.%m.%Y}")
        out(fig, "level_change_by_category")
        out_table(pd.DataFrame({"before, M/day": (before / 1e6).round(2), "after, M/day": (after / 1e6).round(2),
                                "Δ, M/day": (delta / 1e6).round(2),
                                "Δ, %": ((after / before.replace(0, np.nan) - 1) * 100).round(1),
                                "share of the change, %": (delta / d_tot * 100).round(1) if d_tot else np.nan}
                               ).loc[delta.index], "level_change_by_category")
    else:
        print("No level change found")
else:
    print(f"Level-change analysis needs ≥{2 * W} days of events data")

### 4.6 Spike days

A spike day has more than `SPIKE_RATIO` above its baseline — the median of the same weekday over the previous four
weeks, so a Monday is compared with Mondays. Each bar is one spike day (the last 30 are shown), split into the
categories that were above their own baseline that day.
- One colour dominates the bar — one source flooded: an incident, a misconfiguration, a mass event (updates,
  scanning). Look at that source on that date.
- Many colours — general activity was high (end of month, a business event).
- The same category spiking again and again — its baseline is wrong: it is a new normal, or it floods regularly and is
  a candidate for filtering.
- The table `spikes` lists every spike with the baseline, the excess in % and the main category.

*Example (illustrative):* 29.09–01.10 are spikes of +40 %, each bar mostly Windows (+300 M a day) → three days of a mass
event on workstations; if it was patch deployment, expect the same after the next patch day.

In [ ]:
# --- 4.6 spike days: which categories made them ---
if HAS_CAT and tot.notna().sum() >= 5 * 7:
    base = weekday_baseline(evd)
    tot_b = base.sum(axis=1, min_count=1)
    is_spike = (tot > tot_b * (1 + SPIKE_RATIO)) & tot_b.notna()
    spike_days = tot.index[is_spike][-30:]
    if len(spike_days):
        excess = (evd - base).loc[spike_days].clip(lower=0).fillna(0)
        order = excess.sum().sort_values(ascending=False)
        order = order[order > 0].index
        colors = palette(evd.columns)
        labels = [f"{d:%d.%m.%Y}<br>{d:%a}" for d in spike_days]
        fig = go.Figure()
        for cat in order:
            fig.add_trace(go.Bar(x=labels, y=excess.loc[spike_days, cat] / 1e6, name=cat, marker_color=colors[cat],
                                 hovertemplate="%{x}<br>" + cat + ": +%{y:,.1f} M<extra></extra>"))
        fig.update_layout(template="plotly_white", barmode="stack", height=500, yaxis_title="M events above baseline",
                          title=f"Spike days (> {SPIKE_RATIO:.0%} above the same weekday of the last 4 weeks): "
                                f"what was extra", legend=dict(orientation="h", yanchor="top", y=-0.35))
        fig.update_xaxes(tickangle=0, type="category")
        out(fig, "spikes")
        top_cat = excess.idxmax(axis=1)
        out_table(pd.DataFrame({"events, M": (tot[spike_days] / 1e6).round(1),
                                "baseline, M": (tot_b[spike_days] / 1e6).round(1),
                                "above baseline, %": ((tot[spike_days] / tot_b[spike_days] - 1) * 100).round(0),
                                "main category": top_cat,
                                "its extra, M": (excess.max(axis=1) / 1e6).round(1)}, index=spike_days), "spikes")
    else:
        print(f"No spike days above {SPIKE_RATIO:.0%} of the same-weekday baseline")
else:
    print("Spike analysis needs ≥5 weeks of events data")

### 4.7 Weekly profile: people or machines

Each row is a category, each column a weekday; the colour is that weekday's volume in % of the category's average
workday (white = 100 %, red = less, blue = more). Rows are sorted by the weekend / workday ratio.
- **Ratio ≥ `WEEKEND_MACHINE`** — the category barely notices the weekend: background flow generated by machines
  (firewalls, DNS, servers, agents). It does not depend on how many people work, and a lot of noise lives here —
  the first place to look for events to cut.
- **Ratio ≤ `WEEKEND_HUMAN`** — the flow follows working hours: workstations, proxy, authentication. Connecting more
  workstations grows exactly this flow, with the same weekly rhythm.
- The table `weekly_profile` gives the ratio, the class and the average workday volume.

*Example (illustrative):* DNS 0.97 and Firewall 0.92 — background; Proxy 0.35 and AAA 0.40 — people. Half of the
DNS volume on a Sunday comes from servers talking to each other; that is where filtering is cheap.

In [ ]:
# --- 4.7 weekly profile: people vs machines ---
if HAS_CAT:
    rec = evd.dropna(how="all").tail(TREND_WINDOW)
    main_c = [c for c in grp.columns if c in rec.columns]          # categories with ≥1 % share (section 4)
    prof = rec[main_c].groupby(rec.index.dayofweek).mean()
    if set(range(7)) <= set(prof.index):
        workday = prof.loc[0:4].mean()
        rel = prof.div(workday) * 100
        wend = (prof.loc[[5, 6]].mean() / workday).sort_values()
        kind = pd.Series(np.where(wend >= WEEKEND_MACHINE, "machine (background)",
                         np.where(wend <= WEEKEND_HUMAN, "people (working hours)", "mixed")), index=wend.index)
        days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
        fig = go.Figure(go.Heatmap(z=rel.loc[range(7), wend.index].T.values, x=days, y=list(wend.index),
                                   colorscale="RdBu", zmid=100, zmin=0, zmax=200,
                                   colorbar=dict(title="% of an<br>average workday"),
                                   hovertemplate="%{y}, %{x}: %{z:.0f} % of an average workday<extra></extra>"))
        fig.update_layout(template="plotly_white", height=max(350, 34 * len(wend) + 120),
                          title=f"Weekly profile per category (last {len(rec)} days)")
        out(fig, "weekly_profile")
        out_table(pd.DataFrame({"weekend / workday": wend.round(2), "flow": kind,
                                "workday, M/day": (workday[wend.index] / 1e6).round(2)}), "weekly_profile")
    else:
        print("Weekly profile needs every weekday in the data")
else:
    print("Not enough events data (need ≥14 days)")

### 4.8 Stability and silences

**The map** — one row per category, one column per day; the colour is that day's volume divided by the category's
usual level for that weekday (median of the same weekday over the previous 4 weeks, so a quiet Sunday is
compared with Sundays): white ≈ normal, **red = less, dark red = (almost) nothing**,
blue = more. Hover a cell for the date, the ratio and the events.
- A red vertical stripe through all rows — the whole collection stopped (an outage of the SIEM or the collectors).
- A red cell or a short red run in one row — one source type went silent: its collection task, agent or the source
  itself. This is the history of collection reliability per source type.
- Blue cells — floods; compare with section 4.6.

**The table** `category_stability`:
- `day-to-day variation, %` — how much the category jumps from day to day once its weekly rhythm is removed.
  Steady categories are predictable; jumpy ones cause capacity surprises and need a margin.
- `silent days` / `last silent day` — days below `SILENCE_RATIO` of the usual level.

*Example (illustrative):* Firewall shows two dark red days on 05–06.08 and nothing else → the firewall collection task
stopped for two days and nobody noticed; this is what source-flow monitoring in the SIEM should alert on.
Proxy variation 35 % vs 5 % for the others → size its share with a margin.

In [ ]:
# --- 4.8 stability and silences: is every category arriving every day? ---
if HAS_CAT and tot.notna().sum() >= 21:
    med = weekday_baseline(evd)       # usual level = the same weekday of the previous 4 weeks
    ratio = (evd / med.replace(0, np.nan)).clip(upper=2)
    cats = list(evd.dropna(how="all").tail(28).mean().sort_values(ascending=False).index)
    fig = go.Figure(go.Heatmap(z=ratio[cats].T.values, x=ratio.index, y=cats, zmin=0, zmax=2, zmid=1,
                               colorscale=[[0, "rgb(178,24,43)"], [0.25, "rgb(244,165,130)"], [0.5, "rgb(247,247,247)"],
                                           [1, "rgb(33,102,172)"]],
                               colorbar=dict(title="× usual<br>level"),
                               customdata=evd[cats].T.values / 1e6,
                               hovertemplate="%{y}<br>%{x|" + HOVER_DAY + "}<br>%{z:.2f} × usual level"
                                             "<br>%{customdata:,.2f} M events<extra></extra>"))
    style(fig, "Collection map: each category's daily volume vs its usual level (red = missing, blue = above)",
          "", height=max(380, 30 * len(cats) + 160), slider=True)
    fig.update_layout(hovermode="closest")
    out(fig, "collection_map")

    silent = (ratio < SILENCE_RATIO) & (med > 0)
    rec = evd.dropna(how="all").tail(28)
    # weekly rhythm removed using the same 28 days, so an older level change does not count as variation
    resid = rec / rec.groupby(rec.index.dayofweek).transform("mean")
    stab = pd.DataFrame({
        "share, %": (rec.mean() / rec.mean().sum() * 100).round(2),
        "day-to-day variation, %": (resid.std() / resid.mean() * 100).round(1),
        "silent days": silent.sum(),
        "last silent day": silent.apply(lambda s: s[s].index.max()),
    }).loc[cats]
    out_table(stab, "category_stability")
else:
    print("Stability analysis needs ≥3 weeks of events data")

## 5. Disk cost per category

NNLS regression `bytes = Σ bᵢ × eventsᵢ` (non-negative bytes per event per category).
`p5…p95` comes from bootstrap: a narrow interval means the coefficient is reliable.
With only a few weeks of data the intervals are wide — trust them after 3+ months of history.

In [ ]:
if HAS_CAT:
    yb = daily.loc[grp.index, "bytes"].values.astype(float)
    A = grp.values.astype(float)
    coef, _ = nnls(A, yb)
    mape_fit = np.mean(np.abs(yb - A @ coef) / yb)

    rng = np.random.default_rng(0)
    boot = np.array([nnls(A[i], yb[i])[0]
                     for i in tqdm([rng.integers(0, len(yb), len(yb)) for _ in range(500)], desc="bootstrap")])

    cost = pd.DataFrame({
        "share_%": (grp.sum() / grp.sum().sum() * 100).values,
        "B/event": coef,
        "p5": np.percentile(boot, 5, axis=0),
        "p95": np.percentile(boot, 95, axis=0),
        "GiB/day": gb(grp.tail(28).mean().values * coef),
    }, index=grp.columns).sort_values("GiB/day", ascending=False)

    base = grp.tail(28).mean().reindex(cost.index)
    fig = go.Figure(go.Bar(
        x=cost.index, y=cost["GiB/day"], marker_color="steelblue",
        error_y=dict(type="data", symmetric=False,
                     array=gb(base * (cost["p95"] - cost["B/event"])).values,
                     arrayminus=gb(base * (cost["B/event"] - cost["p5"])).values),
        customdata=np.c_[cost["B/event"], cost["p5"], cost["p95"], cost["share_%"]],
        hovertemplate="%{x}<br>%{y:,.1f} GiB/day<br>%{customdata[0]:,.0f} B/event "
                      "(p5 %{customdata[1]:,.0f} – p95 %{customdata[2]:,.0f})<br>share %{customdata[3]:.1f} %<extra></extra>"))
    fig.update_layout(template="plotly_white", height=450, yaxis_title="GiB/day",
                      title="Estimated disk usage per category (whiskers: p5…p95)")
    out(fig, "category_cost")
    print(f"Days in model: {len(yb)}, fit error (MAPE): {mape_fit:.2%}")
    out_table(cost.round(1), "category_cost")

    # events share vs disk share: which categories are heavy per event
    ev28 = grp.tail(28).mean().reindex(cost.index)
    ev_share = ev28 / ev28.sum() * 100
    disk_share = cost["GiB/day"] / cost["GiB/day"].sum() * 100
    fig = go.Figure([go.Bar(x=cost.index, y=ev_share, name="share of events, %", marker_color="lightsteelblue",
                            hovertemplate="%{x}: %{y:.1f} % of events<extra></extra>"),
                     go.Bar(x=cost.index, y=disk_share, name="share of disk, %", marker_color="steelblue",
                            hovertemplate="%{x}: %{y:.1f} % of disk<extra></extra>")])
    fig.update_layout(template="plotly_white", barmode="group", height=430, yaxis_title="%",
                      title="Share of events vs share of disk (last 28 days)",
                      legend=dict(orientation="h", yanchor="top", y=-0.2))
    out(fig, "category_weight")

### How to read section 5

**The chart** — how many GiB per day each category takes on disk, over the last 28 days. Whiskers are the p5…p95
range of the estimate; hover a bar for bytes per event and its range.
- A short whisker — the estimate is reliable and can be used to decide what to filter.
- A long whisker — too little history to separate this category from the others; trust it after 3+ months of data.
- `other <1%` merges small categories; its coefficient is usually unstable.

**The table** `category_cost`: share of events, bytes per event (`B/event`, `p5`, `p95`) and GiB per day.
A category with a small share but a high `B/event` is expensive per event — a candidate for filtering or tuning.
The fit error printed under the chart (MAPE) shows how well the categories explain the daily bytes: a few % is good.

**Share of events vs share of disk** — for each category two bars: its share of events and its share of the disk.
A disk bar much taller than the events bar marks a heavy category (long raw messages): per event removed it frees
the most space. A disk bar much lower is a light category — cutting it barely helps the disk.

*Example (illustrative):* DNS has 7 % of events but 180 B/event and 9 GiB/day with a short whisker — cutting its noise
pays off on disk. `Other` shows 900 B/event with p5 0 … p95 2 500 — not enough data, do not act on it yet.

## 6. Forecast models and backtest

Three models forecast daily bytes:

- **Linear** — linear trend + day-of-week effects over the last `TREND_WINDOW` days. Transparent, good for steady growth.
- **Holt-Winters** — exponential smoothing with damped trend and weekly seasonality. Adapts to level shifts faster.
- **SARIMAX** — seasonal ARIMA `SARIMAX_ORDER` × `SARIMAX_SEASONAL_ORDER` (weekly period) with drift,
  fitted on the last `SARIMAX_WINDOW` days (statsmodels, skipped if not installed).
  Models how day-to-day deviations carry over and the weekly cycle; the drift continues the average growth of the window.

Each model is trained without the last `BACKTEST_DAYS` days and scored on them.
The model with the lowest MAPE is used in sections 7–10.

In [ ]:
def model_linear(y, horizon, window=TREND_WINDOW):
    """Linear trend + day-of-week dummies. Returns yhat and residual sigma."""
    h = y.iloc[-window:]
    def X(idx, t0):
        t = (idx - t0).days.values.astype(float)
        dow = np.eye(7)[idx.dayofweek][:, 1:]
        return np.c_[np.ones(len(idx)), t, dow]
    t0 = h.index[0]
    beta, *_ = np.linalg.lstsq(X(h.index, t0), h.values, rcond=None)
    sigma = np.std(h.values - X(h.index, t0) @ beta)
    fidx = pd.date_range(y.index[-1] + pd.Timedelta(days=1), periods=horizon)
    return pd.Series(X(fidx, t0) @ beta, index=fidx), sigma


def _hw_run(y, m, a, b, g, phi, horizon=0):
    """Additive Holt-Winters with damped trend; returns one-step fitted values and forecast."""
    n = len(y)
    l, tr = y[:m].mean(), (y[m:2 * m].mean() - y[:m].mean()) / m
    S = np.empty(n + m); S[:m] = y[:m] - l
    fit = np.empty(n)
    for i in range(n):
        fit[i] = l + phi * tr + S[i]
        l_new = a * (y[i] - S[i]) + (1 - a) * (l + phi * tr)
        tr = b * (l_new - l) + (1 - b) * phi * tr
        S[i + m] = g * (y[i] - l_new) + (1 - g) * S[i]
        l = l_new
    k = np.arange(1, horizon + 1)
    damp = np.cumsum(phi ** k)
    fc = l + damp * tr + S[n + (k - 1) % m]
    return fit, fc


def model_holt_winters(y, horizon, m=7):
    """Fit alpha/beta/gamma/phi by minimising one-step SSE."""
    v = y.values.astype(float)
    sc = v.mean()
    z = v / sc
    obj = lambda p: np.sum((z[m:] - _hw_run(z, m, *p)[0][m:]) ** 2)
    res = minimize(obj, x0=[0.3, 0.05, 0.1, 0.95], method="L-BFGS-B",
                   bounds=[(0.01, 1), (0.0, 0.5), (0.0, 1), (0.8, 0.999)])
    fit, fc = _hw_run(z, m, *res.x, horizon=horizon)
    sigma = np.std(z[m:] - fit[m:]) * sc
    fidx = pd.date_range(y.index[-1] + pd.Timedelta(days=1), periods=horizon)
    return pd.Series(fc * sc, index=fidx), sigma


try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    HAS_SARIMAX = True
except ImportError:
    HAS_SARIMAX = False
    print("statsmodels not installed: SARIMAX is skipped (pip install statsmodels)")


def model_sarimax(y, horizon, window=SARIMAX_WINDOW,
                  order=SARIMAX_ORDER, seasonal_order=SARIMAX_SEASONAL_ORDER):
    """Seasonal ARIMA with drift on scaled values. Returns yhat and residual sigma.
    trend="c" in statsmodels SARIMAX acts on the differenced series, so with d=1 it is a drift (linear growth)."""
    h = y.iloc[-window:] if window else y
    sc = h.mean()
    z = h.values.astype(float) / sc
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")   # convergence warnings on short / flat series
        res = SARIMAX(z, order=order, seasonal_order=seasonal_order, trend="c").fit(disp=False)
    burn = order[1] + seasonal_order[1] * seasonal_order[3] + seasonal_order[3]   # start-up residuals are not errors
    sigma = np.std(res.resid[burn:]) * sc
    fidx = pd.date_range(y.index[-1] + pd.Timedelta(days=1), periods=horizon)
    return pd.Series(res.forecast(horizon) * sc, index=fidx), sigma


MODELS = {"Linear": model_linear, "Holt-Winters": model_holt_winters}
if HAS_SARIMAX:
    MODELS["SARIMAX"] = model_sarimax

In [ ]:
# --- backtest on the last BACKTEST_DAYS days ---
train, test = y_bytes.iloc[:-BACKTEST_DAYS], y_bytes.iloc[-BACKTEST_DAYS:]
test_ok = ~daily["anomaly"].iloc[-BACKTEST_DAYS:].values   # score only real (non-outage) days

bt, bt_pred = {}, {}
for name, fn in tqdm(MODELS.items(), desc="backtest"):
    pred, _ = fn(train, BACKTEST_DAYS)
    err = (pred.values - test.values)[test_ok]
    bt_pred[name] = pred
    bt[name] = {"MAE, GiB/day": gb(np.mean(np.abs(err))),
                "MAPE, %": np.mean(np.abs(err) / test.values[test_ok]) * 100,
                "bias, GiB/day": gb(np.mean(err)),
                "sigma_bt": np.std(err)}
bt = pd.DataFrame(bt).T.sort_values("MAPE, %")
BEST = bt.index[0]

shown = y_bytes.iloc[-BACKTEST_DAYS * 3:]
outage = daily["anomaly"].reindex(shown.index).fillna(False).astype(bool).values   # interpolated, not real data
fig = go.Figure()
fig.add_trace(line(shown.index, gb(shown), "actual", "GiB", mode="lines+markers",
                   line=dict(color="black", width=1.5), marker=dict(size=5)))
fig.add_trace(line(shown.index[outage], gb(shown[outage]), "outage day, interpolated", "GiB", mode="markers",
                   marker=dict(symbol="x", size=10, color="red")))
for name, pr in bt_pred.items():
    fig.add_trace(line(pr.index, gb(pr), f"{name}: MAPE {bt.loc[name, 'MAPE, %']:.1f}%, "
                                         f"bias {bt.loc[name, 'bias, GiB/day']:+.1f} GiB/day", "GiB",
                       line=dict(width=3.5 if name == BEST else 1.5)))
weekends(fig, shown.index)
mark(fig, train.index[-1], "train end")
style(fig, f"Backtest on the last {BACKTEST_DAYS} days — best: {BEST}", "GiB/day", height=560, daily_ticks=True)
out(fig, "backtest")
out_table(bt.drop(columns="sigma_bt").round(2), "backtest")

### How to read the backtest chart

**What is on it**
- **Black line with dots** — actual GiB per day. Left of the dotted line *train end*: the end of the data the models
  were trained on (only the last weeks are drawn). Right of it: the last `BACKTEST_DAYS` days, hidden from the models.
- **Coloured lines** — what each model predicted for those hidden days, knowing only the data left of the dotted line.
  The thick line is the best model (lowest MAPE).
- **Red ×** — an outage day: its value is interpolated, not real data.
- **Grey bands** — weekends. Every day has a label; hover shows the date with the weekday and every line's value.

**How it is computed**
- The daily series is split in two: everything except the last `BACKTEST_DAYS` days is the training part,
  the last `BACKTEST_DAYS` days are the test part.
- Each model is fitted on the training part only and asked to forecast the test days; the forecast is compared with
  what actually happened.
- Outage days (section 1) are interpolated in the series, and in the test window they are **not scored**:
  an interpolated value is not a real observation, so it must not make a model look better or worse.
- The model with the lowest MAPE becomes the "best" one: its forecast drives sections 7–10,
  and its MAPE sets the width of the uncertainty band in section 7.

**The table under the chart** (one row per model, sorted by MAPE)
- `MAE, GiB/day` — mean absolute error: how many GiB per day the model misses on an average day, in either direction.
- `MAPE, %` — the same miss relative to the actual volume of that day. It compares models and periods of different
  scale; this is the ranking criterion.
- `bias, GiB/day` — mean of *forecast − actual*. Negative = the model under-predicts, positive = over-predicts.
  MAE says how big the misses are, bias says whether they lean one way. A large MAE with bias near 0 is random noise;
  bias close to MAE means the model is systematically off and its level is wrong.

**What to check, in this order**
1. **Level.** Does the best model run through the middle of the black line, or stay above / below it?
   Column `bias` in the table: negative = the model under-predicts. For capacity planning under-prediction is the
   dangerous direction — the disk fills and rotation starts earlier than forecast.
2. **Weekly rhythm.** The model's dips should line up with the actual weekend dips. If they do, the weekly cycle is
   captured. Misaligned dips mean the calendar pattern changed (holidays, shifts).
3. **Spikes.** Days far above every model. No model predicts one-off events: find what happened on those days
   (new source, incident, mass event) in sections 4–5. Spikes that keep repeating are a new normal, not noise.
4. **Size of the error.** MAPE × current daily volume = the typical miss per day in GiB.
   Under ~10 % — fit for planning; 10–20 % — plan with the band from section 7, not the line;
   above 20 % — the flow changed recently and the long-term forecast is unreliable.
5. **Agreement.** Models that are close in the test window can still diverge far apart in section 7.
   The backtest checks the next weeks only; it says nothing about a 180-day horizon.

**Example** *(illustrative numbers, not a real installation)*

> The training data shows a steady ~40 GiB/day that jumps to ~60 GiB/day three weeks before *train end*.
> In the test window:
> - **Linear** — MAPE 8 %, bias −2 GiB/day: follows the level and the weekend dips, slightly low. Best model.
> - **Holt-Winters** — MAPE 12 %, bias +3: the same shape, slightly high.
> - **SARIMAX** — MAPE 19 %, bias +9: above the actual on almost every day — it took the recent jump for a trend
>   and keeps climbing.
> - Two weekdays in the first test week reach ~85 GiB/day and no model saw them coming → look up in sections 4–5
>   which categories grew on those days.
>
> **Conclusion:** for the next weeks expect about 60 ± 5 GiB/day (8 % of 60), and keep a margin because the best
> model runs low. The jump in the training data is recent, so read the 180-day forecast in section 7 as a range,
> not a line, and re-check it in 2–3 weeks.

## 7. Daily volume forecast

Thick line = best model, dashed = other models (all as 7-day means). Shaded band = range between the models, widened by the best model's backtest error.
Read it as: *"if the flow keeps its current dynamics, a typical day will cost this much"*.
If the models diverge strongly, the history contains a recent regime change — trust the short-term part only
and re-check after 2–3 weeks.

In [ ]:
fc = {}
for name, fn in tqdm(MODELS.items(), desc="forecast"):
    fc[name] = fn(y_bytes, HORIZON)[0].clip(lower=0)

def band(fcs, mape):
    """Uncertainty band: range between models widened by the best model's backtest MAPE.
    Smoothed weekly, because capacity depends on sums of many days, not on single-day noise."""
    df_ = pd.DataFrame(fcs).rolling(7, min_periods=1).mean()
    return (df_.min(axis=1) * (1 - mape)).clip(lower=0), df_.max(axis=1) * (1 + mape)

FC = fc[BEST]
FC_LO, FC_HI = band(fc, bt.loc[BEST, "MAPE, %"] / 100)

fig = go.Figure()
fig.add_trace(line(daily.index, gb(daily["bytes"]), "actual", "GiB", line=dict(color="lightgrey", width=1)))
fig.add_trace(line(y_bytes.index, gb(y_bytes.rolling(7).mean()), "actual, 7-day mean", "GiB",
                   line=dict(color="black", width=1.5)))
fig.add_trace(go.Scatter(x=FC.index, y=gb(FC_HI), mode="lines", line=dict(width=0), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=FC.index, y=gb(FC_LO), mode="lines", line=dict(width=0), fill="tonexty",
                         fillcolor="rgba(70,130,180,0.18)", name="uncertainty band", hoverinfo="skip"))
for name, pr in fc.items():
    fig.add_trace(line(pr.index, gb(pr.rolling(7, min_periods=1).mean()), f"{name}, 7-day mean", "GiB",
                       line=dict(width=3 if name == BEST else 1.5, dash="solid" if name == BEST else "dash")))
mark(fig, last_day)
style(fig, f"Daily volume: history and {HORIZON}-day forecast (best: {BEST})", "GiB/day", height=520, slider=True)
out(fig, "volume_forecast")

now7 = y_bytes.tail(7).mean()
end30 = FC.tail(30).mean()
print(f"Now (7-day mean): {gb(now7):.0f} GiB/day ≈ {now7 / D_FACT / 86400:,.0f} EPS")
print(f"In {HORIZON} days ({BEST}): {gb(end30):.0f} GiB/day ≈ {end30 / D_FACT / 86400:,.0f} EPS "
      f"(band {gb(FC_LO.tail(30).mean()):.0f}…{gb(FC_HI.tail(30).mean()):.0f})")

In [ ]:
# --- EPS forecast (what the monitoring team sees in SIEM) ---
y_eps = daily["eps"].mask(daily["anomaly"])
if y_eps.notna().sum() >= 4 * 7:
    y_eps = y_eps.interpolate(limit_direction="both").loc[y_eps.first_valid_index():]
    p_eps = {k: f(y_eps, HORIZON)[0] for k, f in tqdm(MODELS.items(), desc="EPS forecast")}
    lo_e, hi_e = band(p_eps, bt.loc[BEST, "MAPE, %"] / 100)
    fig = go.Figure()
    fig.add_trace(line(y_eps.index, y_eps, "actual EPS", "EPS", line=dict(color="lightgrey", width=1)))
    fig.add_trace(line(y_eps.index, y_eps.rolling(7).mean(), "7-day mean", "EPS", line=dict(color="black")))
    fig.add_trace(go.Scatter(x=hi_e.index, y=hi_e, mode="lines", line=dict(width=0), showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=lo_e.index, y=lo_e, mode="lines", line=dict(width=0), fill="tonexty",
                             fillcolor="rgba(70,130,180,0.18)", name="uncertainty band", hoverinfo="skip"))
    for k, pr in p_eps.items():
        fig.add_trace(line(pr.index, pr.rolling(7, min_periods=1).mean(), f"{k}, 7-day mean", "EPS",
                           line=dict(width=3 if k == BEST else 1.5, dash="solid" if k == BEST else "dash")))
    mark(fig, last_day)
    style(fig, "EPS: history and forecast", "events per second", height=520, slider=True)
    out(fig, "eps_forecast")
else:
    print("EPS forecast needs ≥4 weeks of events data")

### How to read section 7

**Daily volume** (and the same for **EPS**): light grey — actual days, black — 7-day mean of history; right of
*last complete day* — the forecast of every model as a 7-day mean (thick = best model from section 6) and the blue band.
- The band is the spread between the models widened by the best model's backtest error. **Plan with the band, not the
  line**: the upper edge is the pessimistic case for the disk.
- A narrow band — the models agree, the history is steady. A band that fans out — the models disagree about the future,
  usually because the history contains a recent step. Then trust the first 2–4 weeks and watch which line the actual
  follows.
- A straight steep line of `Linear` after a recent step in the history: it treats the step as a trend and may
  overestimate the long term; a flat `Holt-Winters` is the opposite, conservative case.
- Use *3 mo* to see the joint between history and forecast; hover shows each model's value on a given day.

The two printed lines give today's 7-day mean and the expected volume at the end of the horizon with the band.
EPS is derived as bytes / D — without events data D is the vendor value and EPS is only a rough estimate.

*Example (illustrative):* now 60 GiB/day; in 180 days Linear 95, Holt-Winters 62, SARIMAX 75, band 55…105 → the next
month is ~60–65 GiB/day for sure; the half-year can be anything from flat to +60 %. Size the disk for ~100 and re-check
monthly which line the actual follows.

In [ ]:
# --- events per day by category: history and forecast ---
FC_CAT = None
if HAS_CAT and len(ec) >= 4 * 7:
    yc = grp.reindex(daily.index)
    yc.loc[daily["anomaly"].values] = np.nan
    yc = yc.loc[yc.dropna(how="all").index[0]:].interpolate(limit_direction="both")
    can_bt = len(yc) >= BACKTEST_DAYS + 28
    rows_f, fcat = {}, {}
    for cat in tqdm(yc.columns, desc="category forecasts"):
        y = yc[cat]
        if not y.sum():
            continue
        best_m, best_mape = BEST, np.nan
        if can_bt:   # pick the model per category by its own backtest
            tr, te_ = y.iloc[:-BACKTEST_DAYS], y.iloc[-BACKTEST_DAYS:]
            sc = {m: np.mean(np.abs(fn(tr, BACKTEST_DAYS)[0].values - te_.values) / np.maximum(te_.values, 1)) * 100
                  for m, fn in MODELS.items()}
            best_m = min(sc, key=sc.get)
            best_mape = sc[best_m]
        f = MODELS[best_m](y, HORIZON)[0].clip(lower=0)
        fcat[cat] = f
        now = y.tail(7).mean()
        rows_f[cat] = {"model": best_m, "backtest MAPE, %": round(best_mape, 1), "now, M/day": now / 1e6,
                       "in 30 days, M/day": f.iloc[23:30].mean() / 1e6, "in 90 days, M/day": f.iloc[83:90].mean() / 1e6,
                       f"in {HORIZON} days, M/day": f.tail(7).mean() / 1e6,
                       "change, %": (f.tail(7).mean() / now - 1) * 100 if now else np.nan}
    FC_CAT = pd.DataFrame(fcat)
    order = FC_CAT.tail(7).mean().sort_values(ascending=False).index
    colors = palette(evd.columns)
    hist = yc.rolling(7, min_periods=1).mean()
    fig = go.Figure()
    for cat in order:
        col = colors.get(cat, "#888888")
        fig.add_trace(go.Scatter(x=hist.index, y=hist[cat] / 1e6, name=cat, legendgroup=cat, stackgroup="history",
                                 line=dict(width=0.5, color=col), fillcolor=rgba(col, 0.75), hovertemplate="%{y:,.1f} M"))
        fig.add_trace(go.Scatter(x=FC_CAT.index, y=FC_CAT[cat].rolling(7, min_periods=1).mean() / 1e6,
                                 name=cat + " (forecast)", legendgroup=cat, showlegend=False, stackgroup="forecast",
                                 line=dict(width=0.5, color=col, dash="dot"), fillcolor=rgba(col, 0.3),
                                 hovertemplate="%{y:,.1f} M (forecast)"))
    mark(fig, last_day)
    style(fig, f"Events per day by category: history and {HORIZON}-day forecast (7-day means, stacked)",
          "M events/day", height=560, slider=True)
    out(fig, "events_forecast")
    tbl = pd.DataFrame(rows_f).T.loc[order]
    num = [c for c in tbl.columns if c != "model"]
    tbl[num] = tbl[num].astype(float).round(2)
    tot_row = tbl[[c for c in num if c.endswith("M/day")]].sum()
    tbl.loc["total"] = {**tot_row.round(2).to_dict(), "model": "", "backtest MAPE, %": np.nan,
                        "change, %": round((tot_row.iloc[-1] / tot_row.iloc[0] - 1) * 100, 1)}
    out_table(tbl, "events_forecast")
else:
    print("Events forecast needs ≥4 weeks of events data")

### How to read the events forecast

Left of *last complete day* — events per day of each main category (7-day means, stacked, so the top edge is the
total); right of it — each category's forecast, pale and dotted. Click a category in the legend to hide it; hover
shows every category's value on that day.
- Each category gets its own model: the one with the lowest error on its own last `BACKTEST_DAYS` days
  (columns `model`, `backtest MAPE, %` in the table). A high MAPE marks a category whose forecast is unreliable.
- The table `events_forecast` gives now / in 30 / 90 / `HORIZON` days and the change in % per category, and the total.
- The sum of category forecasts can differ from the EPS forecast above: there one model forecasts the total, here
  every category is forecast on its own. A big difference means growth is concentrated in a few categories.
- Read it as "which categories will make the future flow": the top rows of the table by `change, %` × size are the
  ones to plan for — or to optimise first.

*Example (illustrative):* total 700 → 900 M events/day in 180 days (+29 %); Operating system 300 → 420 M (+40 %,
model Linear, MAPE 9 %), Firewall flat, Proxy 60 → 110 M but MAPE 35 % → plan for the operating systems' growth;
treat the Proxy number as a guess until its history settles.

## 8. Capacity: rotation start, retention depth, physical free space

Tailcutter keeps **sum of partitions ≤ DBSPACE × disk** by deleting the oldest days, and also drops days older than TTL.

- **Chart 1 — occupied space.** Grey dashed = what partitions would occupy without size-based rotation;
  blue = expected occupancy with rotation; red = limit. The point where grey crosses red is when automatic deletion starts.
- **Chart 2 — retention depth.** How many days of events will be kept on each future date.
  This is the main business metric: it falls when the flow grows.
- **Chart 3 — physical free space** (`df avail`), forecast = partitions forecast × `df/partitions` ratio,
  plus a plain linear trend of `df` snapshots for comparison.

In [ ]:
def project(stored_series, forecast, limit, ttl, other=0.0):
    """Simulate tailcutter on stored partitions + forecast. Returns frame per future day."""
    vals = np.r_[stored_series.values, forecast.values].astype(float)
    cs = np.r_[0.0, np.cumsum(vals)]
    n0 = len(stored_series)
    rows = []
    for j in range(n0, len(vals)):
        end = j + 1
        start_ttl = max(0, end - ttl)
        no_size_rot = cs[end] - cs[start_ttl] + other
        start = start_ttl
        if limit is not None:
            start = max(start, int(np.searchsorted(cs, cs[end] - (limit - other), side="left")))
        rows.append({"no_size_rotation": no_size_rot,
                     "occupied": cs[end] - cs[start] + other,
                     "depth_days": end - start})
    return pd.DataFrame(rows, index=forecast.index)

if LIMIT is None:
    print("No disk total (no df snapshot and DISK_TOTAL_OVERRIDE is None) — capacity section skipped")
else:
    proj = {k: project(stored_events, s, LIMIT, TTL_DAYS, stored_other)
            for k, s in tqdm({"mid": FC, "lo": FC_LO, "hi": FC_HI}.items(), desc="capacity")}
    P = proj["mid"]
    hit = P.index[P["no_size_rotation"] > LIMIT]
    HIT_DATE = hit[0] if len(hit) else None
    DAYS_TO_LIMIT = (HIT_DATE - last_day).days if HIT_DATE is not None else None
    hit_lo = proj["hi"].index[proj["hi"]["no_size_rotation"] > LIMIT]
    hit_hi = proj["lo"].index[proj["lo"]["no_size_rotation"] > LIMIT]

    # chart 1: occupied space
    fig = go.Figure()
    fig.add_trace(line(disk.index, disk["partitions"] / TiB, "partitions (actual)", "TiB", mode="lines+markers",
                       line=dict(color="black")))
    fig.add_trace(line(P.index, P["no_size_rotation"] / TiB, "without size rotation", "TiB",
                       line=dict(color="grey", dash="dash")))
    fig.add_trace(line(P.index, P["occupied"] / TiB, "with rotation (expected)", "TiB", line=dict(color="royalblue", width=3)))
    fig.add_hline(y=LIMIT / TiB, line_color="red", line_dash="dash",
                  annotation_text=f"limit {DBSPACE:.0%} = {LIMIT/TiB:.2f} TiB", annotation_position="bottom left")
    fig.add_hline(y=DISK_TOTAL / TiB, line_color="black", line_width=1,
                  annotation_text="disk total", annotation_position="top left")
    if HIT_DATE is not None:
        fig.add_shape(type="line", x0=HIT_DATE, x1=HIT_DATE, y0=0, y1=1, yref="paper", line=dict(color="red", width=1))
        fig.add_annotation(x=HIT_DATE, y=LIMIT / TiB, text=f"rotation starts ~{HIT_DATE:%d.%m.%Y}", showarrow=True,
                           arrowhead=2, font=dict(color="red"))
    mark(fig, last_day)
    style(fig, "Occupied by partitions", "TiB", slider=True)
    out(fig, "capacity_occupied")

    # chart 2: retention depth
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=P.index, y=proj["lo"]["depth_days"], mode="lines", line=dict(width=0),
                             showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=P.index, y=proj["hi"]["depth_days"], mode="lines", line=dict(width=0), fill="tonexty",
                             fillcolor="rgba(65,105,225,0.18)", name="uncertainty band", hoverinfo="skip"))
    fig.add_trace(line(P.index, P["depth_days"], f"expected ({BEST})", "days", line=dict(color="royalblue", width=3)))
    fig.add_hline(y=TTL_DAYS, line_color="grey", line_dash="dot", annotation_text=f"TTL {TTL_DAYS}",
                  annotation_position="top left")
    mark(fig, last_day, f"now: {len(stored_events)} days stored")
    style(fig, "Retention depth forecast: days of events kept", "days kept", slider=True)
    out(fig, "capacity_retention")

    # chart 3: physical free space
    phys_avail = DISK_TOTAL - P["occupied"] * RATIO
    fig = go.Figure()
    fig.add_trace(line(disk.index, disk["avail"] / TiB, "df avail (actual)", "TiB", mode="lines+markers",
                       line=dict(color="black")))
    fig.add_trace(line(P.index, phys_avail / TiB, "forecast from partitions", "TiB", line=dict(color="royalblue", width=3)))
    if len(disk) >= 3:
        td = (disk.index - disk.index[0]).total_seconds().values / 86400
        k, c0 = np.polyfit(td, disk["avail"].values, 1)
        tf = (P.index - disk.index[0]).total_seconds().values / 86400
        fig.add_trace(line(P.index, (c0 + k * tf) / TiB, f"linear trend of df = if rotation does not work "
                                                         f"({gb(k):+.0f} GiB/day)", "TiB",
                           line=dict(color="orange", dash="dash")))
    fig.add_hline(y=DISK_TOTAL * (1 - DBSPACE) / TiB, line_color="red", line_dash="dash",
                  annotation_text="free space at limit", annotation_position="top left")
    fig.add_hline(y=0, line_color="black", line_width=1)
    mark(fig, last_day)
    style(fig, "Physical free space", "TiB", slider=True)
    out(fig, "capacity_free_space")

    out_table(pd.DataFrame({
        "expected, TiB": (P["occupied"] / TiB).round(2),
        "without size rotation, TiB": (P["no_size_rotation"] / TiB).round(2),
        "retention, days": P["depth_days"],
        "retention low, days": proj["hi"]["depth_days"],
        "retention high, days": proj["lo"]["depth_days"],
        "free space forecast, TiB": (phys_avail / TiB).round(2),
    }), "capacity_projection", show=False)

    print(f"Disk {DISK_TOTAL/TiB:.2f} TiB, limit {LIMIT/TiB:.2f} TiB, partitions now {stored_total/TiB:.2f} TiB "
          f"({stored_total/LIMIT:.0%} of limit)")
    if HIT_DATE is not None:
        rng_txt = (f"{hit_lo[0]:%d.%m.%Y}" if len(hit_lo) else "?") + "…" + \
                  (f"{hit_hi[0]:%d.%m.%Y}" if len(hit_hi) else f"after {HORIZON} days")
        print(f"Size rotation starts ~{HIT_DATE:%d.%m.%Y} (in {DAYS_TO_LIMIT} days; band {rng_txt})")
    else:
        print(f"Size limit is not reached within {HORIZON} days")
    print(f"Retention depth: now {len(stored_events)} days → in {HORIZON} days ~{P['depth_days'].iloc[-1]} days "
          f"(band {proj['hi']['depth_days'].iloc[-1]}…{proj['lo']['depth_days'].iloc[-1]})")

### How to read section 8

**Occupied by partitions.** Black — actual sum of partitions at each snapshot; grey dashed — what partitions would
occupy if tailcutter deleted only by TTL; blue — expected occupancy with size rotation; red — the limit
`DBSPACE × disk`; black solid — the whole disk.
- Where the grey line crosses red, size rotation starts deleting the oldest days (red vertical line and date).
- **Black already above red** — the limit is exceeded now. Either rotation does not work, or `DBSPACE` in the notebook
  differs from the real tailcutter setting; the "starts in N days" message is meaningless in that case.

**Retention depth** — how many days of events will be kept on each future date (blue), with the band from the
forecast. This is the business number: compare it with the required retention. It falls as the flow grows.
The vertical line shows how many days are stored now.

**Physical free space** — blue: expected `df avail` with rotation working; orange dashed: the plain trend of `df`
snapshots, i.e. the scenario "rotation does not work" — where it crosses zero the disk is full.
The orange line is only meaningful after a few weeks of snapshots.

**The table** `capacity_projection` (saved, not shown): per future day — occupancy with and without size rotation,
retention depth with its band and the free-space forecast.

*Example (illustrative):* limit 8.5 TiB, partitions 7.9 TiB, grey crosses red on 14.03 → in ~5 months rotation starts;
retention falls from 240 to ~150 days by the end of the horizon, band 120…190. If the required retention is 180 days,
the decision (more disk, filtering, or accepting less) is due before the band's lower edge crosses 180.

## 9. What-if: flow changes

**Table:** a uniform change of the whole flow → daily volume and steady-state retention depth.
**Function `what_if_category`:** change of one category, priced with the coefficients from section 5.

In [ ]:
eps_now = now7 / D_FACT / 86400
rows = []
for pct in tqdm([-50, -30, -20, -10, 0, 10, 20, 30, 50, 100], desc="scenarios"):
    day_b = now7 * (1 + pct / 100)
    r = {"flow change, %": pct,
         "ΔEPS": eps_now * pct / 100,
         "GiB/day": gb(day_b),
         "ΔGiB/day": gb(day_b - now7)}
    if LIMIT:
        r["retention, days"] = min(TTL_DAYS, int((LIMIT - stored_other) // day_b))
    rows.append(r)
out_table(pd.DataFrame(rows).set_index("flow change, %").round(1), "what_if")


def what_if_category(category, pct):
    """Daily volume change if one category changes by pct %."""
    if not HAS_CAT or category not in cost.index:
        print(f"Not in model: {category}. Available: {list(cost.index) if HAS_CAT else []}")
        return
    ev = grp[category].tail(28).mean() * pct / 100
    r = cost.loc[category]
    msg = (f"{category} {pct:+}% → {ev:+,.0f} events/day → {gb(ev * r['B/event']):+.1f} GiB/day "
           f"(band {gb(ev * r['p5']):+.1f}…{gb(ev * r['p95']):+.1f})")
    if LIMIT:
        new = now7 + ev * r["B/event"]
        msg += f"; retention ~{min(TTL_DAYS, int((LIMIT - stored_other) // new))} days"
    print(msg)


if HAS_CAT:
    what_if_category(cost.drop(index="other <1%", errors="ignore").index[0], -30)

### How to read section 9

**The table** `what_if`: one row per uniform change of the whole flow. `ΔEPS` and `ΔGiB/day` — the change against today's
7-day mean; `retention, days` — the steady-state depth once the disk is at the limit with that flow
(capped at `TTL_DAYS`).
- Read it as "what does +X % of flow cost in days of retention" — the trade-off between connecting new sources and
  keeping history. The negative rows show what filtering buys.

**`what_if_category("<category>", pct)`** prints the same for one category, priced with its own bytes per event from
section 5 and its p5…p95 range. Call it in a new cell for any category from the section 5 table.

*Example (illustrative):* row +20 % → 72 GiB/day, retention 150 days instead of 180: connecting the planned sources
costs a month of history. `what_if_category("DNS server", -50)` → −4.5 GiB/day, retention ~+12 days: filtering half
of DNS buys back almost two weeks.

## 10. Daily summary

`OK` — nothing to do. `WARN` — look at the referenced section today. `CRIT` — act now (see METHODOLOGY.md).

In [ ]:
status = []
def add(level, check, value, section):
    status.append({"level": level, "check": check, "value": value, "see": section})

# data freshness
age_h = (pd.Timestamp.now() - last_snap).total_seconds() / 3600
add("OK" if age_h < 26 else "WARN", "latest snapshot age", f"{age_h:.0f} h", "1")
ev_last = daily["events"].last_valid_index()
add("OK" if ev_last == last_day else "WARN", "events collected up to",
    f"{ev_last:%d.%m.%Y}" if ev_last is not None else "none", "1")

# yesterday volume vs same weekday baseline (last 4 same weekdays)
same = daily["bytes"].loc[daily.index.dayofweek == last_day.dayofweek].iloc[-5:-1]
base = same.median()
yb_ = daily["bytes"].iloc[-1]
dev = (yb_ - base) / base if base else np.nan
add("OK" if abs(dev) < WARN_VOLUME_DEV else "WARN", f"volume {last_day:%d.%m} vs same weekday",
    f"{gb(yb_):.0f} GiB ({dev:+.0%})", "7")

# per-category deviation (source dropout or flood)
if HAS_CAT and last_day in ev_cat.index:
    hist_c = ev_cat.loc[ev_cat.index.dayofweek == last_day.dayofweek].iloc[-5:-1].median()
    cur_c = ev_cat.loc[last_day]
    rel = ((cur_c - hist_c) / hist_c.replace(0, np.nan)).dropna()
    rel = rel[hist_c.reindex(rel.index) > ev_cat.loc[last_day].sum() * 0.001]   # ignore tiny categories
    bad = rel[rel.abs() >= WARN_CATEGORY_DEV]
    add("OK" if bad.empty else "WARN", "categories off baseline",
        ", ".join(f"{k} {v:+.0%}" for k, v in bad.sort_values().items()) or "none", "4")

# D drift
if HAS_D and pd.notna(daily["D"].iloc[-1]):
    d_dev = daily["D"].iloc[-1] / D_FACT - 1
    add("OK" if abs(d_dev) < WARN_D_DRIFT else "WARN", "D yesterday vs 28-day median",
        f"{daily['D'].iloc[-1]:.0f} B/event ({d_dev:+.0%})", "2")

# rotation since the previous snapshot (3b)
if ROT is not None and len(ROT):
    r = ROT.iloc[-1]
    oldest = f"{r['oldest day']:%d.%m.%Y}" if pd.notna(r["oldest day"]) else "?"
    add("WARN" if str(r["reason"]).startswith("NOT") else "OK", "deleted since the previous snapshot",
        f"{r['days deleted']} days, {r['freed, GiB']:.0f} GiB {r['reason']}; oldest day {oldest}, "
        f"{r['days stored']} days stored", "3b")
if ROT_LEVEL is not None:
    add("WARN" if abs(ROT_LEVEL / 100 - DBSPACE) > 0.03 else "OK", "observed rotation level vs DBSPACE",
        f"deletes at ~{ROT_LEVEL:.0f} % of disk; DBSPACE = {DBSPACE:.0%}", "3b")

# events without a category
if NN_SHARE is not None:
    add("WARN" if NN_SHARE > WARN_NOT_NORMALIZED else "OK", "events without a category, 28 days",
        f"{NN_SHARE:.1%}", "4.3")
if LEVEL_CHANGE is not None:
    add("OK", "largest level change in events", f"~{LEVEL_CHANGE[0]:%d.%m.%Y}: {LEVEL_CHANGE[1] / 1e6:+,.0f} M/day ({LEVEL_CHANGE[2]:+.0%})", "4.5")

# capacity
if LIMIT:
    occ = stored_total / LIMIT
    add("CRIT" if occ > 1.02 else ("WARN" if occ > 0.95 else "OK"), "partitions / limit", f"{occ:.0%}", "8")
    if DAYS_TO_LIMIT is not None:
        add("WARN" if DAYS_TO_LIMIT < WARN_DAYS_TO_LIMIT else "OK", "size rotation starts in",
            f"{DAYS_TO_LIMIT} days (~{HIT_DATE:%d.%m.%Y})", "8")
    add("OK", "retention now → horizon", f"{len(stored_events)} → {P['depth_days'].iloc[-1]} days", "8")
    free = disk["avail"].iloc[-1] / DISK_TOTAL
    add("CRIT" if free < 0.05 else ("WARN" if free < 1 - DBSPACE else "OK"), "physical free space",
        f"{free:.1%} ({disk['avail'].iloc[-1]/TiB:.2f} TiB)", "8")

# rotation health
if "tailcutter_cron" in health:
    add("OK" if health["tailcutter_cron"] == 1 else "CRIT", "tailcutter in cron",
        "yes" if health["tailcutter_cron"] == 1 else "NO", "METHODOLOGY")
if "tailcutter_log_age_min" in health:
    a = health["tailcutter_log_age_min"]
    add("OK" if 0 <= a < WARN_TC_LOG_AGE else "WARN", "tailcutter log age",
        "no log" if a < 0 else f"{a:.0f} min", "METHODOLOGY")

add("OK", "best forecast model", f"{BEST} (backtest MAPE {bt.loc[BEST, 'MAPE, %']:.1f}%)", "6")

summary = pd.DataFrame(status)
colors = {"OK": "background-color:#d9f2d9;color:#000", "WARN": "background-color:#fff2b3;color:#000",
          "CRIT": "background-color:#f8c6c6;color:#000"}
out_table(summary, "summary", colors=colors, color_col="level")

### How to read section 10

One row per check: `level` (OK / WARN / CRIT), what was checked, its value and the section to open (`see`).
Go top-down through every row that is not OK; the meaning of each check and what to do is in `METHODOLOGY.md`, section 4.
A CRIT that contradicts another row (for example *partitions / limit* CRIT while the disk is stable and tailcutter
runs) usually means a setting in section 0 does not match the server — check it before acting.
The table is also saved as `summary_<day>.xlsx`, so the status of different days can be compared.

## 11. Compared with previous days

Is the picture getting better or worse? The latest values against the previous day, the same weekday a week ago and
the 28-day mean — for the disk, the flow and the notebook's own forecasts.
**Needs data for every day.** If a day is missing, the nearest earlier one is used; the dates actually compared are
printed above the table.

**Key results of every run** (forecast, rotation start, retention at the horizon, D, …) are saved as
`key_metrics_<day>.xlsx` in `EXPORT_DIR`; this section reads all of them back, so it can show how the notebook's own
forecast moved. Keep the folder — it is the memory of the previous runs.

**The comparison table** `comparison`: one row per metric.
- `latest` / `previous` / `week ago` / `28-day mean` and the change against each.
- `verdict` compares with a week ago (the same weekday, so weekly rhythm does not count): **better ▲ / worse ▼**
  for metrics with a good direction (more free space, fewer events without a category, a later rotation start),
  plain ↑ / ↓ for neutral ones (flow volume), `≈` for changes under 2 %.
- Groups: **disk** (by snapshot), **flow** (by day of events), **forecast** (by notebook run).

**Category changes** `category_changes`: the categories that changed most against the same weekday a week ago.

**Mini-charts** — the last 28 points of the key metrics, the latest one red: is today an ordinary day or an outlier?

**Forecast drift** — how the forecast of rotation start, retention at the horizon and the volume at the horizon moved
from run to run. A rotation date that comes closer every run is the clearest sign the picture is getting worse.

*Example (illustrative):* free space 0.9 → 0.7 TiB (worse ▼), days stored 250 → 249, events without a category 12 → 15 %
(worse ▼), Proxy +40 % vs last Tuesday; forecast: size rotation 15.03 a week ago, 01.03 yesterday, 22.02 today →
the flow grows faster than the forecast assumed; look at sections 4.5 and 4.6 for the cause.

In [ ]:
# --- 11. compared with previous days ---
G = globals().get
# 1. key results of this run (read back by later runs)
key_now = pd.DataFrame([{
    "GiB/day, 7-day mean": round(gb(now7), 2),
    "D, B/event": round(D_FACT, 1) if HAS_D else np.nan,
    "events without a category, %": round(NN_SHARE * 100, 2) if NN_SHARE is not None else np.nan,
    "best model": BEST, "backtest MAPE, %": round(bt.loc[BEST, "MAPE, %"], 2),
    "forecast GiB/day at the horizon": round(gb(end30), 2),
    "size rotation starts": G("HIT_DATE") if LIMIT else None,
    "days to size rotation": G("DAYS_TO_LIMIT") if LIMIT else None,
    "retention at the horizon, days": G("P")["depth_days"].iloc[-1] if LIMIT else None,
    "days stored": len(stored_events),
}], index=pd.DatetimeIndex([REPORT_DAY], name="report day"))
out_table(key_now, "key_metrics", show=False)
runs = (pd.concat([pd.read_excel(f, index_col=0) for f in sorted(EXPORT_DIR.glob("key_metrics_*.xlsx"))])
        .pipe(lambda d: d[~d.index.duplicated(keep="last")]).sort_index())
runs.index = pd.to_datetime(runs.index)
runs = runs.loc[:REPORT_DAY]


# 2. series to compare: (group, metric, series, good direction: +1 higher is better, -1 lower is better, 0 neutral)
def masked(s):
    return s.mask(daily["anomaly"])


def n80(row):
    row = row.dropna().sort_values(ascending=False)
    return int(((row.cumsum() / row.sum()) < 0.8).sum()) + 1 if row.sum() > 0 else np.nan


snap_days = disk.index if len(disk) else pd.DatetimeIndex([])
series = [
    ("disk", "df used, TiB", disk["used"] / TiB if len(disk) else None, -1),
    ("disk", "df free, TiB", disk["avail"] / TiB if len(disk) else None, +1),
    ("disk", "partitions, TiB", disk["partitions"] / TiB if len(disk) else None, -1),
    ("disk", "days stored", ROT["days stored"] if ROT is not None else None, +1),
    ("disk", "deleted, GiB", ROT["freed, GiB"] if ROT is not None else None, 0),
    ("disk", "added, GiB", ROT["added, GiB"] if ROT is not None else None, 0),
    ("flow", "GiB/day", masked(gb(daily["bytes"])), 0),
    ("flow", "M events/day", masked(daily["events"]) / 1e6, 0),
    ("flow", "EPS", masked(daily["eps"]), 0),
    ("flow", "D, B/event", masked(daily["D"]), -1),
    ("flow", "events without a category, %",
     (evd[NOT_NORMALIZED] / evd.sum(axis=1, min_count=1) * 100) if NOT_NORMALIZED in evd.columns else None, -1),
    ("flow", "categories making 80 %", evd.apply(n80, axis=1) if HAS_CAT else None, 0),
    ("forecast", "forecast GiB/day at the horizon", runs["forecast GiB/day at the horizon"], -1),
    ("forecast", "days to size rotation", pd.to_numeric(runs["days to size rotation"], errors="coerce"), +1),
    ("forecast", "retention at the horizon, days", pd.to_numeric(runs["retention at the horizon, days"], errors="coerce"), +1),
    ("forecast", "backtest MAPE, %", runs["backtest MAPE, %"], -1),
]


def points(s, by_day):
    """latest, previous, a week ago and the 28-point mean before the latest, with the timestamps used."""
    s = s.dropna()
    if not len(s):
        return [np.nan] * 4, [None] * 3
    if by_day:          # flow: fixed calendar days relative to the last complete day
        t0 = REPORT_DAY
        s = s.loc[:t0]
        if not len(s):
            return [np.nan] * 4, [None] * 3
        t_latest = s.index[-1]
    else:
        t_latest = s.index[-1]
    prev = s.loc[:t_latest - pd.Timedelta(seconds=1)]
    week = s.loc[:t_latest - pd.Timedelta(days=7) + pd.Timedelta(hours=12)]
    mean = s.loc[:t_latest - pd.Timedelta(seconds=1)].tail(28)
    vals = [s.iloc[-1], prev.iloc[-1] if len(prev) else np.nan, week.iloc[-1] if len(week) else np.nan,
            mean.mean() if len(mean) else np.nan]
    return vals, [t_latest, prev.index[-1] if len(prev) else None, week.index[-1] if len(week) else None]


def pct(a, b):
    return (a / b - 1) * 100 if pd.notna(a) and pd.notna(b) and b else np.nan


rows, used = [], {}
for group, name, s, good in series:
    if s is None:
        continue
    (lat, prv, wk, mn), ts = points(s, by_day=(group == "flow"))
    if pd.isna(lat):
        continue
    used.setdefault(group, ts)
    ref = wk if pd.notna(wk) else prv
    ch = pct(lat, ref)
    if pd.isna(ch):
        verdict = ""
    elif abs(ch) < 2:
        verdict = "≈"
    elif good == 0:
        verdict = "↑" if ch > 0 else "↓"
    else:
        verdict = "better ▲" if ch * good > 0 else "worse ▼"
    rows.append({"group": group, "metric": name, "latest": lat, "previous": prv, "Δ vs previous": lat - prv,
                 "week ago": wk, "Δ vs week ago, %": ch, "28-point mean": mn, "Δ vs mean, %": pct(lat, mn),
                 "verdict": verdict})
cmp_tbl = pd.DataFrame(rows).set_index(["group", "metric"]).round(2)

fmt = lambda t: "—" if t is None else (f"{t:%d.%m.%Y %H:%M}" if t.hour or t.minute else f"{t:%d.%m.%Y}")
for g, ts in used.items():
    print(f"{g:9s}: latest {fmt(ts[0])}, previous {fmt(ts[1])}, week ago {fmt(ts[2])}")
colour = {"better ▲": "background-color:#d9f2d9;color:#000", "worse ▼": "background-color:#f8c6c6;color:#000"}
out_table(cmp_tbl, "comparison", colors=colour, color_col="verdict")

# 3. category changes against the same weekday a week ago
if HAS_CAT and REPORT_DAY in evd.index and REPORT_DAY - pd.Timedelta(days=7) in evd.index:
    a, b = evd.loc[REPORT_DAY], evd.loc[REPORT_DAY - pd.Timedelta(days=7)]
    chg = pd.DataFrame({f"{REPORT_DAY:%d.%m}, M": a / 1e6, f"{REPORT_DAY - pd.Timedelta(days=7):%d.%m}, M": b / 1e6,
                        "Δ, M": (a - b) / 1e6, "Δ, %": (a / b.replace(0, np.nan) - 1) * 100})
    chg = chg.reindex(chg["Δ, M"].abs().sort_values(ascending=False).index).round(2)
    print(f"\nCategories against the same weekday a week ago ({REPORT_DAY:%a}):")
    out_table(chg.head(10), "category_changes")

# 4. mini-charts: the last 28 points of the key metrics, the latest one red
mini = [(n, s) for g, n, s, _ in series if s is not None and s.dropna().size >= 2
        and n in ("df free, TiB", "days stored", "deleted, GiB", "GiB/day", "M events/day", "D, B/event",
                  "events without a category, %", "days to size rotation", "retention at the horizon, days")]
if mini:
    cols = 3
    rws = -(-len(mini) // cols)
    fig = make_subplots(rows=rws, cols=cols, subplot_titles=[n for n, _ in mini], vertical_spacing=0.12)
    for k, (n, s) in enumerate(mini):
        s = s.dropna().tail(28)
        r, c = k // cols + 1, k % cols + 1
        fig.add_trace(go.Scatter(x=s.index, y=s, mode="lines+markers", marker=dict(size=4), line=dict(color="steelblue"),
                                 showlegend=False, hovertemplate="%{x|%d.%m.%Y %H:%M}<br>%{y:,.2f}<extra>" + n + "</extra>"),
                      row=r, col=c)
        fig.add_trace(go.Scatter(x=[s.index[-1]], y=[s.iloc[-1]], mode="markers", marker=dict(size=10, color="red"),
                                 showlegend=False, hovertemplate="latest: %{y:,.2f}<extra>" + n + "</extra>"),
                      row=r, col=c)
    fig.update_xaxes(tickformat="%d.%m")
    fig.update_layout(template="plotly_white", height=260 * rws + 60, title="Key metrics, last 28 points (red = latest)")
    out(fig, "comparison_minicharts")

# 5. forecast drift from run to run
if len(runs) >= 2:
    fig = make_subplots(rows=1, cols=3, subplot_titles=("size rotation starts (date)",
                                                        f"retention in {HORIZON} days, days",
                                                        f"forecast in {HORIZON} days, GiB/day"))
    rs = pd.to_datetime(runs["size rotation starts"], errors="coerce")
    fig.add_trace(go.Scatter(x=runs.index, y=rs, mode="lines+markers", showlegend=False,
                             hovertemplate="run %{x|%d.%m.%Y}: rotation ~%{y|%d.%m.%Y}<extra></extra>"), row=1, col=1)
    fig.add_trace(go.Scatter(x=runs.index, y=pd.to_numeric(runs["retention at the horizon, days"], errors="coerce"),
                             mode="lines+markers", showlegend=False,
                             hovertemplate="run %{x|%d.%m.%Y}: %{y} days<extra></extra>"), row=1, col=2)
    fig.add_trace(go.Scatter(x=runs.index, y=runs["forecast GiB/day at the horizon"], mode="lines+markers",
                             showlegend=False, hovertemplate="run %{x|%d.%m.%Y}: %{y:,.1f} GiB/day<extra></extra>"),
                  row=1, col=3)
    fig.update_xaxes(tickformat="%d.%m", title_text="notebook run (last complete day)")
    fig.update_yaxes(tickformat="%d.%m.%Y", row=1, col=1)
    fig.update_layout(template="plotly_white", height=420, title="Forecast drift: how the forecast moved from run to run")
    out(fig, "forecast_drift")
else:
    print("\nForecast drift starts from the second daily run (key_metrics files of earlier days in EXPORT_DIR)")